# Data Model Installer

Install a Databricks **Industry Data Model** (catalog, schemas, tables, foreign keys,
governance tags, metric views, and optional sample data) into a Unity Catalog catalog of
your choice. Model
definitions come live from the
[lakehouse-industry-data-models](https://github.com/databricks-industry-solutions/lakehouse-industry-data-models)
repo, or from a local folder you provide.

**Databricks Serverless compatible** - every operation is a plain `spark.sql` call,
with Python threads for parallelism. No SparkContext, RDDs, caching, or
classic-compute-only APIs.

## Minimal use

**Select an industry in the `1. industry` widget and click `Run All`.** Everything else
has a sensible default (`size = mvm`, catalog = the industry name, 32 threads). On the
very first Run All nothing is selected yet, so the notebook asks you to pick an industry
and Run All again.

## How it runs (launcher pattern)

When you Run All interactively, the notebook **launches the install as a Databricks job**, prints the run URL, then **waits for the
job to finish**, pulsing a liveness line every 5s and pointing at the job URL so you can
monitor it closely. When the job completes it prints the **Catalog Explorer URL**. The job runs the install and tags
itself (prefix `dbx_vibe_agent_installer_`) with the `industry`, `size`, `version`, and
final `duration`. The launched job always runs the install in-place (it never
re-launches itself).

The installer resolves the **latest version** of the model, downloads its SQL, rewrites
the catalog name to yours, splits it into statements, and applies them in dependency
order with live timestamped progress:
`catalog -> schemas -> tables -> foreign keys -> tags -> metric views`.
Failures are captured and retried serially at the end, so transient concurrency errors
self-heal and re-running is safe (idempotent).

> `mvm` = *Minimum Viable Model* (recommended). `ecm` = *Expanded Coverage Model* (full).

## Parameters (widgets)

Nine widgets, shown in order:

| Widget | Meaning |
|---|---|
| **1. industry** | Industry to install (40 choices). Defaults to a placeholder so you choose explicitly. |
| **2. size** | `mvm` (minimal, recommended) or `ecm` (full). |
| **3. catalog** | Base target catalog. Blank = the industry name. Hosts `_metrics` for multi-catalog layouts. |
| **4. cataloging style** | `One Catalog` (default), `Catalog per Domain`, or `Catalog per Division`. |
| **5. catalog prefix** | Optional prefix for satellite catalogs (multi-catalog styles default to `cat_` when blank). |
| **6. catalog suffix** | Optional suffix for satellite catalogs. |
| **7. local install** | Optional local/Volume path: a model folder, its parent version folder, or a `model.json` file (any `*.json`) produced by the agent. When set, the sibling `schemas/` (and `metrics/`) are located automatically and the GitHub download is skipped. |
| **8. generate samples** | `No` (default) or `Yes`. `Yes` populates every installed table with synthetic rows after the structure is in place. |
| **9. sample rows** | Rows per table when samples are on: `5`, `10` (default), `20`, `50`, `100`. Applies to every table. |

Advanced settings are not shown as widgets and use built-in defaults, forwarded to the
launched job automatically: **32 threads x 20-statement batches** (the measured
serverless optimum), metric views **on**, source = this repo @ `main`.

## Sample data

Set **8. generate samples** to `Yes` to fill the installed tables with synthetic rows.
Nothing about the install changes otherwise: samples run last, after tables, foreign
keys, tags, and metric views, and are skipped if the structural install left failures.

What the generated data guarantees:

| Guarantee | How |
|---|---|
| Primary keys are unique | Each table draws from its own key block, composite keys unique as a tuple, every key value in the type its column declares. |
| Every foreign key resolves | Parent keys are minted before any child references them, and each child copies a real parent key (whole tuple for composite keys). Cycles and self-references are ordered so no reference points at a key that does not exist yet. |
| Nothing lands half-broken | An in-memory integrity gate re-checks key uniqueness, foreign-key containment, and NOT NULL columns before the first write. If it fails, **no** table is written. |
| Values look plausible | Column names and types drive the value shape: codes come from a vocabulary, emails look like emails, decimals respect their precision and scale, and date pairs that name an order (`created`/`updated`, `start`/`end`) are generated in that order. |
| Reruns are reproducible | A fixed seed (`sample_seed`) means the same install produces the same rows. |

The structure is read back from `information_schema` after the install, so generation
targets the tables, keys, and relationships Unity Catalog actually holds, not what the
model file declared. Views, metric views, and internal schemas are never populated.

An optional pass asks a Databricks Foundation Model endpoint for realistic value pools
for free-text columns (names, descriptions, cities). It is time-boxed per table and
never used for keys: if an endpoint is slow, unavailable, or answers with garbage, that
table falls back to deterministic values and the install continues.

Advanced sample settings, forwarded to the job like the other advanced defaults:
`sample_seed` (default `20260801`), `sample_llm` (`true`), `sample_llm_endpoints`
(comma-separated, defaults to `databricks-gpt-oss-120b` and
`databricks-meta-llama-3-3-70b-instruct`), and `sample_threads` (`8`).

### Installing an older version (local_install)

The repo serves the latest version. To install an older one: download that version's
folder (e.g. `data-models/<industry>/v2/<model_size>/`, containing `schemas/` and
set **local_install** to that
path. Point it at the model folder, its parent version folder, or the `model.json`
file itself (any `*.json`); the sibling `schemas/` (and `metrics/`) are located
automatically. This is also how you deploy a **Dry Run** output from the agent. When
set, version auto-detection and the GitHub download are skipped.


In [ ]:
# === Widgets (creates the UI parameters) ===
INDUSTRIES = ["advertising", "agriculture", "airlines", "apparel_fashion", "automotive", "banking", "chemical_mfg", "clinical_trials", "construction", "consumer_goods", "ecommerce", "education", "energy_utilities", "food_beverage", "gaming", "genomics_biotech", "grocery", "health_insurance", "healthcare", "legal", "life_insurance", "manufacturing", "media_broadcasting", "mining", "ngo", "oil_gas", "payments_fintech", "pharmaceuticals", "real_estate", "restaurants", "retail", "semiconductors", "shipping_ports", "sports_entertainment", "staffing_hr", "telecommunication", "transport_shipping", "travel_hospitality", "waste_management", "water_utilities"]   # the 40 industries shipped in the repo

SELECT_PROMPT = "(select an industry)"   # forces an explicit choice on the first run

# Ten widgets are shown, in order. Everything else (threads, batch size, metric views,
# source repo/ref, session id, sample tuning) uses the defaults in INSTALLER_DEFAULTS
# below and is forwarded automatically to the launched job, so the UI stays minimal.
dbutils.widgets.removeAll()
dbutils.widgets.dropdown("operation", "Install", ["Install", "Uninstall"], "1. operation")
dbutils.widgets.dropdown("model", SELECT_PROMPT, [SELECT_PROMPT] + INDUSTRIES, "2. industry")
dbutils.widgets.dropdown("model_size", "mvm", ["mvm", "ecm"], "3. size")
dbutils.widgets.text("catalog_name", "", "4. catalog (blank = industry name)")
dbutils.widgets.dropdown("cataloging_style", "One Catalog", ["One Catalog", "Catalog per Division", "Catalog per Domain"], "5. cataloging style")
dbutils.widgets.text("catalog_prefix", "", "6. catalog prefix (optional)")
dbutils.widgets.text("catalog_suffix", "", "7. catalog suffix (optional)")
dbutils.widgets.text("local_install", "", "8. local install (blank = pull from repo)")
dbutils.widgets.dropdown("generate_samples", "No", ["No", "Yes"], "9. generate samples")
dbutils.widgets.dropdown("sample_rows", "10", ["5", "10", "20", "50", "100"], "10. sample rows")

# Advanced settings - intentionally NOT shown as widgets. 32 threads x 20-statement
# batches is the measured serverless optimum; metric views on; source = this repo @ main.
INSTALLER_DEFAULTS = {
    "threads": "32",
    "batch_size": "20",
    "include_metrics": "true",
    "source_repo": "databricks-industry-solutions/lakehouse-industry-data-models",
    "source_ref": "main",
    "github_token": "",
    # Sample generation advanced settings. The two visible widgets decide whether to
    # generate and how many rows; these tune HOW, and are forwarded to the job.
    "sample_seed": "20260801",
    "sample_threads": "8",
    "sample_llm": "true",
    "sample_llm_endpoints": "databricks-gpt-oss-120b,databricks-meta-llama-3-3-70b-instruct",
}


In [ ]:
# === Shared helpers ===
import datetime
import threading
import os
import json

INSTALLER_TAG_PREFIX = "dbx_vibe_agent_installer_"

_LOG_LOCK = threading.Lock()
_LOG_BUFFER = []   # mirrored to a UC Volume file so logs are retrievable after the run
_SINK = {"path": None}   # set by setup_log_sink(); log() appends to it inline


def log(msg):
    """Print a timestamped log line (streams live in the cell) and append it to the
    Volume log sink INLINE on the calling thread.

    Inline append (vs a background flush daemon) is deliberate: during a heavy
    spark.sql phase the 32 worker threads + blocked main thread starve any daemon
    thread of the GIL, so a daemon-based flush would not update the file until the
    phase ended. Worker threads call log() between gRPC calls (they hold the GIL
    then), so inline append keeps the file genuinely live during the phase."""
    line = "[%s] %s" % (datetime.datetime.now().strftime("%H:%M:%S"), msg)
    with _LOG_LOCK:
        _LOG_BUFFER.append(line)
        print(line, flush=True)
        path = _SINK["path"]
        if path:
            # Rewrite the whole buffer (truncate+write). UC Volume FUSE does not
            # surface "a" (append) writes to a separate `fs cp` reader, but a full
            # "w" rewrite IS visible, so this keeps the CLI tail genuinely live.
            # Single serialized writer (under _LOG_LOCK) -> always a consistent file.
            try:
                with open(path, "w") as f:
                    f.write("\n".join(_LOG_BUFFER) + "\n")
            except Exception:
                pass


def _flush_log_durable():
    """Persist the full log buffer durably before dbutils.notebook.exit(). The per-line
    open('w') rewrites go through UC Volume FUSE, whose write-back is async, so the
    final rewrite (the one carrying the verdict) can be dropped when the process
    terminates even after os.fsync(). dbutils.fs.put writes synchronously through the
    Volumes API, so the verdict always survives; open('w') is kept for live-tail."""
    with _LOG_LOCK:
        path = _SINK["path"]
        if not path:
            return
        content = "\n".join(_LOG_BUFFER) + "\n"
        try:
            with open(path, "w") as f:
                f.write(content)
                f.flush()
                os.fsync(f.fileno())
        except Exception:
            pass
        try:
            dbutils.fs.put(path, content, True)
        except Exception:
            pass


In [ ]:
# === Core SQL logic (self-contained; no external module imports) ===
"""Core SQL-processing logic for data-model-installer.ipynb.

Self-contained in this notebook cell. No Spark / Databricks dependency.
"""
import re
from collections import OrderedDict


_CATALOGING_STYLE_MAP = {
    "One Catalog": "one_catalog",
    "Catalog per Division": "catalog_per_division",
    "Catalog per Domain": "catalog_per_domain",
    "one_catalog": "one_catalog",
    "catalog_per_division": "catalog_per_division",
    "catalog_per_domain": "catalog_per_domain",
}

_DIVISION_CATALOG_MAP = {
    "operations": "operations",
    "business": "business",
    "corporate": "corporate",
    "supporting": "business",
}

_INTERNAL_SCHEMAS = frozenset({"_metrics", "_install", "_metamodel", "information_schema", "default"})

_DOMAIN_HEADER_RE = re.compile(r"--\s*Schema for Domain:\s*(\S+)", re.IGNORECASE)
_METRIC_DOMAIN_HEADER_RE = re.compile(r"--\s*Metric views for domain:\s*(\S+)", re.IGNORECASE)
_DIVISION_TAG_RE = re.compile(r"dbx_division['\"]?\s*=\s*['\"](\w+)['\"]", re.IGNORECASE)
_CATALOG_SCHEMA_RE = re.compile(r"`([^`]+)`\.`([^`]+)`")
_CREATE_SCHEMA_RE = re.compile(
    r"CREATE\s+(?:OR\s+REPLACE\s+)?(?:DATABASE|SCHEMA)\s+(?:IF\s+NOT\s+EXISTS\s+)?`([^`]+)`\.`([^`]+)`",
    re.IGNORECASE,
)


class CatalogResolver:
    """Mirrors vibe-modelling-agent CatalogResolver (industry-agnostic)."""

    def __init__(self, style, base_catalog, prefix="", suffix=""):
        self.style = _CATALOGING_STYLE_MAP.get(style, style)
        if self.style not in ("one_catalog", "catalog_per_division", "catalog_per_domain"):
            self.style = "one_catalog"
        self.base_catalog = base_catalog
        self.prefix = prefix or ""
        self.suffix = suffix or ""
        if self.style in ("catalog_per_domain", "catalog_per_division") and not self.prefix and not self.suffix:
            self.prefix = "cat_"

    def resolve_catalog(self, domain_dict):
        if self.style == "one_catalog":
            return self.base_catalog
        if self.style == "catalog_per_division":
            division = (domain_dict.get("division") or "business").lower().strip()
            cat = _DIVISION_CATALOG_MAP.get(division, "business")
            return self._apply_affixes(cat)
        if self.style == "catalog_per_domain":
            domain_name = domain_dict.get("domain") or domain_dict.get("name", "")
            cat = _snake_case(domain_name) if domain_name else "default"
            return self._apply_affixes(cat)
        return self.base_catalog

    def all_catalogs(self, domains):
        return sorted({self.resolve_catalog(d) for d in domains})

    def _apply_affixes(self, name):
        if self.prefix or self.suffix:
            return "%s%s%s" % (self.prefix, name, self.suffix)
        return name


def _snake_case(name):
    s = re.sub(r"[^a-zA-Z0-9]+", "_", str(name or "").strip())
    s = re.sub(r"_+", "_", s).strip("_").lower()
    return s or "default"


def normalize_cataloging_style(style):
    return _CATALOGING_STYLE_MAP.get((style or "").strip(), "one_catalog")


def parse_schema_metadata(sql_text, src_catalog=None):
    """Extract schema -> {domain, division} from one schema or metric SQL file."""
    meta = {}
    domain = None
    dm = _DOMAIN_HEADER_RE.search(sql_text) or _METRIC_DOMAIN_HEADER_RE.search(sql_text)
    if dm:
        domain = dm.group(1).strip().lower()

    division = "business"
    div_m = _DIVISION_TAG_RE.search(sql_text)
    if div_m:
        division = div_m.group(1).strip().lower()

    for cat, schema in _CREATE_SCHEMA_RE.findall(sql_text):
        if src_catalog and cat != src_catalog:
            continue
        if schema.startswith("_"):
            continue
        key = schema.lower()
        meta[key] = {"domain": domain or key, "division": division, "schema": schema}

    if not meta:
        for cat, schema in _CATALOG_SCHEMA_RE.findall(sql_text):
            if src_catalog and cat != src_catalog:
                continue
            if schema.startswith("_"):
                continue
            key = schema.lower()
            if key not in meta:
                meta[key] = {"domain": domain or key, "division": division, "schema": schema}

    if domain:
        dk = domain.lower()
        if dk in meta:
            meta[dk]["domain"] = dk

    return meta


def collect_schema_metadata(raw_by_name, src_catalog):
    """Merge per-file schema metadata across all schema + metric SQL sources."""
    merged = OrderedDict()
    for name, raw in raw_by_name.items():
        if not name.endswith(".sql"):
            continue
        for schema_key, info in parse_schema_metadata(raw, src_catalog).items():
            if schema_key not in merged:
                merged[schema_key] = dict(info)
            else:
                if info.get("division") and info["division"] != "business":
                    merged[schema_key]["division"] = info["division"]
    return merged


def build_schema_catalog_map(schema_meta, resolver, src_catalog, base_catalog):
    """Map schema_name -> target_catalog for layout rewrite."""
    mapping = {}
    for schema_key, info in schema_meta.items():
        target = resolver.resolve_catalog(
            {"domain": info.get("domain") or schema_key, "division": info.get("division") or "business"}
        )
        mapping[schema_key] = target
        mapping[info.get("schema", schema_key)] = target

    for internal in _INTERNAL_SCHEMAS:
        mapping[internal] = base_catalog

    if resolver.style == "one_catalog":
        for key in list(mapping.keys()):
            mapping[key] = base_catalog

    return mapping


def target_catalogs_for_layout(schema_catalog_map, base_catalog, style):
    style = normalize_cataloging_style(style)
    if style == "one_catalog":
        return [base_catalog]
    cats = sorted(set(schema_catalog_map.values()))
    if base_catalog not in cats:
        cats.append(base_catalog)
    return sorted(set(cats))


def rewrite_catalog_layout(text, src_catalog, schema_catalog_map, base_catalog):
    """Rewrite `src`.`schema` triples using per-schema target catalogs."""
    if not src_catalog or not text:
        return text

    pairs = []
    seen = set()
    for m in _CATALOG_SCHEMA_RE.finditer(text):
        cat, schema = m.group(1), m.group(2)
        if cat != src_catalog:
            continue
        key = (cat, schema)
        if key in seen:
            continue
        seen.add(key)
        if schema.startswith("_") or schema.lower() in _INTERNAL_SCHEMAS:
            target = base_catalog
        else:
            target = schema_catalog_map.get(schema, schema_catalog_map.get(schema.lower(), base_catalog))
        pairs.append((cat, schema, target))

    pairs.sort(key=lambda t: len(t[1]), reverse=True)
    out = text
    for cat, schema, target in pairs:
        out = out.replace("`%s`.`%s`" % (cat, schema), "`%s`.`%s`" % (target, schema))
    return out


def split_sql(text):
    """Split a SQL script into individual statements on top-level ';'."""
    stmts = []
    buf = []
    i = 0
    n = len(text)
    in_line_comment = False
    in_block_comment = False
    in_squote = False
    in_btick = False
    in_dollar = False
    while i < n:
        c = text[i]
        nxt = text[i + 1] if i + 1 < n else ''
        if in_line_comment:
            if c == '\n':
                in_line_comment = False
                buf.append(c)
            i += 1
            continue
        if in_block_comment:
            if c == '*' and nxt == '/':
                in_block_comment = False
                i += 2
                continue
            i += 1
            continue
        if in_dollar:
            if c == '$' and nxt == '$':
                buf.append('$$')
                in_dollar = False
                i += 2
                continue
            buf.append(c)
            i += 1
            continue
        if in_squote:
            buf.append(c)
            if c == "'":
                if nxt == "'":
                    buf.append(nxt)
                    i += 2
                    continue
                in_squote = False
            i += 1
            continue
        if in_btick:
            buf.append(c)
            if c == '`':
                in_btick = False
            i += 1
            continue
        if c == '-' and nxt == '-':
            in_line_comment = True
            i += 2
            continue
        if c == '/' and nxt == '*':
            in_block_comment = True
            i += 2
            continue
        if c == '$' and nxt == '$':
            in_dollar = True
            buf.append('$$')
            i += 2
            continue
        if c == "'":
            in_squote = True
            buf.append(c)
            i += 1
            continue
        if c == '`':
            in_btick = True
            buf.append(c)
            i += 1
            continue
        if c == ';':
            stmt = ''.join(buf).strip()
            if stmt:
                stmts.append(stmt)
            buf = []
            i += 1
            continue
        buf.append(c)
        i += 1
    tail = ''.join(buf).strip()
    if tail:
        stmts.append(tail)
    return stmts


def detect_catalog_token(catalogs_sql):
    m = re.search(r"CREATE\s+CATALOG\s+(?:IF\s+NOT\s+EXISTS\s+)?`([^`]+)`",
                  catalogs_sql, re.IGNORECASE)
    return m.group(1) if m else None


def rewrite_catalog(text, src_token, dst_token):
    if not src_token or src_token == dst_token:
        return text
    return text.replace("`%s`" % src_token, "`%s`" % dst_token)


def categorize(stmt):
    u = re.sub(r"\s+", " ", stmt).strip().upper()
    if u.startswith("CREATE CATALOG"):
        return "catalog"
    if u.startswith("CREATE DATABASE") or u.startswith("CREATE SCHEMA"):
        return "schema"
    if u.startswith("CREATE OR REPLACE TABLE") or u.startswith("CREATE TABLE") \
            or u.startswith("CREATE EXTERNAL TABLE"):
        return "table"
    if "ADD CONSTRAINT" in u and "FOREIGN KEY" in u:
        return "fk"
    if "SET TAGS" in u or "UNSET TAGS" in u:
        return "tag"
    if ("CREATE OR REPLACE VIEW" in u or u.startswith("CREATE VIEW")
            or "CREATE MATERIALIZED VIEW" in u):
        return "metric"
    return "other"


_TARGET_RE = re.compile(
    r"ALTER\s+(?:TABLE|SCHEMA|VIEW|MATERIALIZED\s+VIEW)\s+(`[^`]+`(?:\.`[^`]+`){0,2})",
    re.IGNORECASE)


def target_key(stmt):
    m = _TARGET_RE.search(stmt)
    if not m:
        return ""
    parts = m.group(1).split("`.`")
    cleaned = [p.strip("`") for p in parts]
    return ".".join(cleaned)


def build_batches(statements, batch_size, group_by_target=False):
    if not group_by_target:
        return [statements[i:i + batch_size]
                for i in range(0, len(statements), batch_size)]
    groups = OrderedDict()
    for s in statements:
        k = target_key(s)
        groups.setdefault(k, []).append(s)
    return [g for g in groups.values()]


_SET_TAGS_RE = re.compile(
    r"^(?P<prefix>.*?\bSET\s+TAGS\s*\()(?P<body>.*)\)\s*$",
    re.IGNORECASE | re.DOTALL)


def merge_tag_statements(tag_stmts):
    merged = OrderedDict()
    passthrough = []
    for s in tag_stmts:
        m = _SET_TAGS_RE.match(s.strip())
        if not m:
            passthrough.append(s)
            continue
        prefix = re.sub(r"\s+", " ", m.group("prefix")).strip()
        body = m.group("body").strip()
        merged.setdefault(prefix, []).append(body)
    out = ["%s%s)" % (prefix, ", ".join(bodies)) for prefix, bodies in merged.items()]
    return out + passthrough


def build_layout_context(raw_by_name, src_token, base_catalog, cataloging_style,
                         catalog_prefix="", catalog_suffix=""):
    """Compute schema catalog map + target catalog list for an install run."""
    style = normalize_cataloging_style(cataloging_style)
    resolver = CatalogResolver(style, base_catalog, catalog_prefix, catalog_suffix)
    schema_meta = collect_schema_metadata(raw_by_name, src_token)
    schema_map = build_schema_catalog_map(schema_meta, resolver, src_token, base_catalog)
    targets = target_catalogs_for_layout(schema_map, base_catalog, style)
    return {
        "style": style,
        "resolver": resolver,
        "schema_meta": schema_meta,
        "schema_catalog_map": schema_map,
        "target_catalogs": targets,
    }


def rewrite_model_sql(raw_text, src_token, layout_ctx):
    """Rewrite one SQL file for the chosen cataloging layout."""
    base = layout_ctx["schema_catalog_map"].get("_metrics") or layout_ctx["target_catalogs"][0]
    if layout_ctx["style"] == "one_catalog":
        return rewrite_catalog(raw_text, src_token, base)
    return rewrite_catalog_layout(raw_text, src_token, layout_ctx["schema_catalog_map"], base)


def filter_catalog_statements(stmts, layout_ctx):
    """Drop source CREATE CATALOG statements when layout creates catalogs at install time."""
    if layout_ctx["style"] == "one_catalog":
        return stmts
    return [s for s in stmts if categorize(s) != "catalog"]


In [ ]:
# === JobLauncher: launch this notebook as a tagged Databricks job (agent pattern) ===
import re as _jl_re


class JobLauncher:
    """Create/reuse a named Databricks job for this notebook and run it, with tags.

    Mirrors the agent's launcher: build with the notebook path, the widget values to
    pass as base_parameters, and a {tag_key: tag_value} dict attached to the job.
    """

    _TAG_SAFE_RE = _jl_re.compile(r"[^A-Za-z0-9._-]")

    @staticmethod
    def _sanitize_tag(value):
        s = JobLauncher._TAG_SAFE_RE.sub("_", str(value))
        s = _jl_re.sub(r"_+", "_", s)
        return s.strip("_").strip(".").strip("-")

    def __init__(self, notebook_path, widget_key_values, job_tags=None):
        self.notebook_path = str(notebook_path)
        self.widget_key_values = {str(k): str(v) for k, v in widget_key_values.items()}
        raw = dict(job_tags or {})
        self.job_tags = {self._sanitize_tag(k): self._sanitize_tag(v) for k, v in raw.items()}

    @staticmethod
    def _detect_compute_type():
        """Detect serverless vs classic (ported from the agent's proven launcher).

        Serverless compute exposes a non-empty `clusterUsageTags.clusterId`, so a
        clusterId-only check misfires and wrongly attaches a classic cluster - which a
        serverless-only workspace rejects ("Only serverless compute is supported").
        The reliable signals are the IS_SERVERLESS env var and the absence of a
        cluster *name*. Returns (is_serverless, cluster_id)."""
        import os as _jl_os
        if _jl_os.environ.get("IS_SERVERLESS", "").upper() == "TRUE":
            return True, None
        try:
            spark.conf.get("spark.databricks.clusterUsageTags.clusterName")
        except Exception:
            return True, None
        try:
            _cid = spark.conf.get("spark.databricks.clusterUsageTags.clusterId", "")
            if _cid:
                return False, _cid
        except Exception:
            pass
        return True, None

    @staticmethod
    def _get_workspace_context():
        host, org = "", ""
        try:
            ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
            try:
                host = ctx.apiUrl().get()
            except Exception:
                pass
            try:
                org = ctx.workspaceId().get()
            except Exception:
                pass
        except Exception:
            pass
        if not host:
            try:
                from databricks.sdk import WorkspaceClient as _WC
                host = str(_WC().config.host)
            except Exception:
                pass
        return (host or "").rstrip("/"), org

    @staticmethod
    def get_current_notebook_path():
        try:
            ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
            try:
                p = ctx.notebookPath().get()
                if p:
                    return p
            except Exception:
                pass
            import json as _j
            c = _j.loads(ctx.toJson())
            for k in ("notebook_path", "notebookPath"):
                v = (c.get("extraContext") or {}).get(k, "") or (c.get("tags") or {}).get(k, "")
                if v:
                    return v
        except Exception:
            pass
        return ""

    def launch(self, job_name=None, run_name=None):
        import time as _t
        from databricks.sdk import WorkspaceClient as _WC
        from databricks.sdk.service import jobs as _jobs
        out = {"success": False, "job_id": None, "run_id": None, "job_url": "", "error": None}
        try:
            w = _WC()
            job_name = job_name or ("dbx_vibe_installer_%d" % int(_t.time()))
            is_serverless, cluster_id = self._detect_compute_type()

            def _build_task(attach_cluster):
                t = _jobs.Task(
                    task_key="install",
                    notebook_task=_jobs.NotebookTask(
                        notebook_path=self.notebook_path,
                        base_parameters=self.widget_key_values),
                    timeout_seconds=14400,
                )
                if attach_cluster and cluster_id:
                    t.existing_cluster_id = cluster_id
                return t

            existing = None
            try:
                for j in w.jobs.list(name=job_name):
                    if j.settings and j.settings.name == job_name:
                        existing = j.job_id
                        break
            except Exception:
                pass

            def _create_and_run(attach_cluster):
                task = _build_task(attach_cluster)
                if existing:
                    w.jobs.reset(job_id=existing,
                                 new_settings=_jobs.JobSettings(name=job_name, tags=self.job_tags, tasks=[task]))
                    jid = existing
                else:
                    jid = w.jobs.create(name=job_name, tags=self.job_tags, tasks=[task]).job_id
                r = w.jobs.run_now(job_id=jid)
                return jid, r

            try:
                job_id, run = _create_and_run(attach_cluster=(not is_serverless))
            except Exception as ce:
                # Defense in depth: if a serverless-only workspace rejects the attached
                # cluster (detection misfired), retry as a pure serverless task.
                if "serverless" in str(ce).lower() and not is_serverless:
                    job_id, run = _create_and_run(attach_cluster=False)
                else:
                    raise
            host, org = self._get_workspace_context()
            url = "%s/jobs/%s/runs/%s%s" % (host, job_id, run.run_id, ("?o=%s" % org if org else "")) if host else ""
            out.update({"success": True, "job_id": job_id, "run_id": run.run_id, "job_url": url})
        except Exception as e:
            out["error"] = str(e)
        return out

    @staticmethod
    def wait_for_run(run_id, job_url="", pulse_seconds=20, logger=None):
        """Block until the launched run reaches a terminal state, emitting a liveness
        timestamped pulse every pulse_seconds (no link). Returns
        {life_cycle_state, result_state, notebook_output, error}."""
        import time as _t
        from databricks.sdk import WorkspaceClient as _WC
        log = logger or (lambda m: print(m, flush=True))
        w = _WC()
        terminal = {"TERMINATED", "INTERNAL_ERROR", "SKIPPED"}
        out = {"life_cycle_state": "", "result_state": "", "notebook_output": "", "error": None}
        start = _t.time()
        while True:
            try:
                run = w.jobs.get_run(run_id=run_id)
            except Exception as e:
                log("   poll error (%s) - retrying in %ds" % (str(e)[:120], pulse_seconds))
                _t.sleep(pulse_seconds)
                continue
            state = run.state if run else None
            st = state.life_cycle_state.value if (state and state.life_cycle_state) else ""
            rs = state.result_state.value if (state and state.result_state) else ""
            elapsed = int(_t.time() - start)
            if st in terminal:
                out["life_cycle_state"], out["result_state"] = st, rs
                try:
                    trid = run.tasks[0].run_id if (run and run.tasks) else run_id
                    ro = w.jobs.get_run_output(run_id=trid)
                    out["notebook_output"] = (ro.notebook_output.result if ro.notebook_output else "") or ""
                    if ro.error:
                        out["error"] = ro.error
                except Exception as e:
                    out["error"] = str(e)
                return out
            log("   [%s] job still running [%s] %ds elapsed"
                % (_t.strftime("%H:%M:%S"), st or "PENDING", elapsed))
            _t.sleep(pulse_seconds)

    @staticmethod
    def update_job_tags(updated_tags):
        """Merge tags onto the job currently running this notebook (best effort)."""
        res = {"success": False, "error": None}
        if not updated_tags:
            res["error"] = "empty"
            return res
        try:
            ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
            job_id_str = ""
            try:
                job_id_str = ctx.jobId().get()
            except Exception:
                pass
            if not job_id_str:
                res["error"] = "not running as a job (no jobId)"
                return res
            from databricks.sdk import WorkspaceClient as _WC
            from databricks.sdk.service import jobs as _jobs
            w = _WC()
            job_id = int(job_id_str)
            info = w.jobs.get(job_id=job_id)
            existing = dict(info.settings.tags or {})
            new = {JobLauncher._sanitize_tag(k): JobLauncher._sanitize_tag(v) for k, v in updated_tags.items()}
            merged = {**existing, **new}
            w.jobs.update(job_id=job_id, new_settings=_jobs.JobSettings(tags=merged))
            res["success"] = True
        except Exception as e:
            res["error"] = str(e)
        return res


In [ ]:
# === Config, model loading (repo or local), and plan building ===
DEFAULT_DDL_THREADS = 8   # UC-throttle-safe concurrency cap for fk/tag phases

import json as _json
import os
import re as _re
import urllib.request
from collections import OrderedDict


def _wget(name, default=""):
    """Read a widget/param value if present, else return default. Advanced settings are
    passed as job parameters (not shown as widgets), so they are read through this safe
    getter rather than dbutils.widgets.get, which raises when a widget was never created
    in an interactive run."""
    try:
        v = dbutils.widgets.get(name)
    except Exception:
        return default
    return v if (v is not None and v != "") else default


def resolve_config():
    """Read widgets into a config dict and log the resolved inputs."""
    industry = dbutils.widgets.get("model").strip()
    model_size = dbutils.widgets.get("model_size").strip()
    D = INSTALLER_DEFAULTS
    cfg = {
        "operation": _wget("operation", "Install").strip().lower() or "install",
        "industry": industry,
        "model_size": model_size,
        "catalog": dbutils.widgets.get("catalog_name").strip() or industry,
        "cataloging_style": _wget("cataloging_style", "One Catalog").strip() or "One Catalog",
        "catalog_prefix": _wget("catalog_prefix", "").strip(),
        "catalog_suffix": _wget("catalog_suffix", "").strip(),
        "local_install": dbutils.widgets.get("local_install").strip(),
        "session_id": _wget("session_id", "").strip(),
        "threads": int(_wget("threads", D["threads"]).strip() or D["threads"]),
        "batch_size": int(_wget("batch_size", D["batch_size"]).strip() or D["batch_size"]),
        "include_metrics": _wget("include_metrics", D["include_metrics"]).strip().lower() == "true",
        "source_repo": _wget("source_repo", D["source_repo"]).strip() or D["source_repo"],
        "source_ref": _wget("source_ref", D["source_ref"]).strip() or D["source_ref"],
        "github_token": _wget("github_token", D["github_token"]).strip(),
        "resolved_version": "unknown",
        "sample": resolve_sample_config(_wget),
    }
    # UC throttles metadata DDL (ADD CONSTRAINT/SET TAGS) at high concurrency, so the
    # fk/tag phases run at a capped concurrency while object creation uses full threads.
    cfg["ddl_threads"] = max(4, min(cfg["threads"], DEFAULT_DDL_THREADS))
    # Accept a model.json (or any *.json) FILE path for local_install, not just a folder: the
    # DDL the installer runs lives in a sibling schemas/ folder next to model.json, so pasting
    # the agent-produced model.json volume path installs from the folder that CONTAINS it.
    # local_install_raw is kept so the launched job re-derives + logs the resolution.
    cfg["local_install_raw"] = cfg["local_install"]
    cfg["local_install"] = _local_install_dir(cfg["local_install"])
    # A local install points at its own model folder, so the industry is only a label
    # and need not be one of the 40 shipped here - that is how a freshly generated
    # model (which no repo folder describes yet) gets installed. Default the label to
    # the folder name so the dropdown does not have to be touched at all.
    if cfg["local_install"]:
        if industry in ("", SELECT_PROMPT):
            industry = os.path.basename(cfg["local_install"].rstrip("/")) or "local_model"
            cfg["industry"] = industry
            if not dbutils.widgets.get("catalog_name").strip():
                cfg["catalog"] = industry
        assert industry, "an industry label is required"
    else:
        # On uninstall the industry is only a label used to locate the install
        # manifest in the target catalog; it need not be a shipped model (a custom
        # vibe-generated model is uninstalled by catalog + manifest, not repo lookup).
        assert industry in INDUSTRIES or cfg["operation"] == "uninstall", \
            "Unknown industry: %s" % industry
    assert cfg["operation"] in ("install", "uninstall"), \
        "operation must be Install or Uninstall, got %r" % cfg["operation"]
    assert model_size in ("mvm", "ecm"), "model_size must be mvm or ecm"
    assert cfg["threads"] >= 1 and cfg["batch_size"] >= 1
    cfg["mode"] = "LOCAL" if cfg["local_install"] else "REPO"
    if normalize_cataloging_style(cfg["cataloging_style"]) != "one_catalog" and not cfg["catalog_prefix"] and not cfg["catalog_suffix"]:
        cfg["catalog_prefix"] = "cat_"
    return cfg


def _http_get(url, token=""):
    req = urllib.request.Request(url)
    req.add_header("Accept", "application/vnd.github+json")
    req.add_header("User-Agent", "data-model-installer")
    if token:
        req.add_header("Authorization", "Bearer %s" % token)
    with urllib.request.urlopen(req, timeout=60) as r:
        return r.read().decode("utf-8")


def _gh_contents(cfg, path):
    url = "https://api.github.com/repos/%s/contents/%s?ref=%s" % (
        cfg["source_repo"], path, cfg["source_ref"])
    return _json.loads(_http_get(url, cfg["github_token"]))


def latest_version(cfg):
    """Highest version folder (v1, v2, ...) available for the industry in the repo."""
    items = _gh_contents(cfg, "data-models/%s" % cfg["industry"])
    versions = sorted(
        (it["name"] for it in items
         if it.get("type") == "dir" and _re.match(r"^v\d+$", it["name"])),
        key=lambda v: int(v[1:]))
    if not versions:
        raise Exception("No version folders (vN) for industry %s" % cfg["industry"])
    return versions[-1]


def _resolve_repo_base(cfg):
    versions = _gh_contents(cfg, "data-models/%s" % cfg["industry"])
    vs = sorted((it["name"] for it in versions
                 if it.get("type") == "dir" and _re.match(r"^v\d+$", it["name"])),
                key=lambda v: int(v[1:]))
    if not vs:
        raise Exception("No version folders (vN) for industry %s" % cfg["industry"])
    cfg["resolved_version"] = vs[-1]
    log("Versions available: %s  -> using %s" % (vs, vs[-1]))
    return "data-models/%s/%s/%s" % (cfg["industry"], vs[-1], cfg["model_size"])


def _local_install_dir(path):
    """Accept EITHER a model folder OR a model.json (any *.json) FILE path for local_install.
    The installer runs the DDL that lives in a sibling schemas/ folder next to model.json, so
    when the operator pastes the agent-produced model.json volume path we install from the
    folder that CONTAINS it. Folder inputs pass through unchanged. alias=installer-local-modeljson-path"""
    p = (path or "").strip()
    if not p:
        return p
    _stripped = p.rstrip("/")
    if _stripped.lower().endswith(".json") or (os.path.isfile(_stripped) and not os.path.isdir(_stripped)):
        d = os.path.dirname(_stripped)
        return d or p
    return p


def _resolve_local_base(cfg):
    p = cfg["local_install"]
    cfg["resolved_version"] = "local"
    _raw = cfg.get("local_install_raw", p)
    if _raw and _raw != p:
        log("[installer-local-modeljson-path FIRED] local_install pointed at a file (%s); "
            "installing from its folder %s (the DDL lives in the sibling schemas/ + metrics/)."
            % (os.path.basename(_raw), p))
    for cand in (p, os.path.join(p, cfg["model_size"])):
        if os.path.isdir(os.path.join(cand, "schemas")):
            return cand
    raise Exception("No 'schemas' folder under %r (tried direct and /%s). Point "
                    "local_install at the model folder (or its model.json) containing "
                    "schemas/ and metrics/." % (p, cfg["model_size"]))


def _list_sql(ref, is_local, cfg):
    if is_local:
        if not os.path.isdir(ref):
            return []
        return [(n, os.path.join(ref, n)) for n in sorted(os.listdir(ref)) if n.endswith(".sql")]
    items = _gh_contents(cfg, ref)
    return [(it["name"], it["download_url"]) for it in items
            if it.get("type") == "file" and it["name"].endswith(".sql")]


def _read_sql(ref, cfg):
    if ref.startswith("http://") or ref.startswith("https://"):
        return _http_get(ref, cfg["github_token"])
    with open(ref, "r") as f:
        return f.read()


def _rank(name):
    if "catalog" in name:
        return (0, name)
    if "foreign_key" in name:
        return (2, name)
    return (1, name)


def build_plan(cfg):
    """Load the model SQL, rewrite the catalog, split, and categorize into phases."""
    is_local = bool(cfg["local_install"])
    if is_local:
        base = _resolve_local_base(cfg)
        schema_files = _list_sql(os.path.join(base, "schemas"), True, cfg)
        metric_files = _list_sql(os.path.join(base, "metrics"), True, cfg) if cfg["include_metrics"] else []
        log("Local base: %s" % base)
    else:
        base = _resolve_repo_base(cfg)
        schema_files = _list_sql(base + "/schemas", False, cfg)
        metric_files = _list_sql(base + "/metrics", False, cfg) if cfg["include_metrics"] else []
    schema_files.sort(key=lambda nv: _rank(nv[0]))
    metric_files.sort(key=lambda nv: nv[0])
    log("Schema files: %d   Metric files: %d" % (len(schema_files), len(metric_files)))

    src_token = None
    raw_by_name = {}
    for name, ref in schema_files + metric_files:
        raw = _read_sql(ref, cfg)
        raw_by_name[name] = raw
        if src_token is None and "catalog" in name:
            src_token = detect_catalog_token(raw)
    if src_token is None:
        for raw in raw_by_name.values():
            src_token = detect_catalog_token(raw)
            if src_token:
                break
    layout_ctx = build_layout_context(
        raw_by_name, src_token, cfg["catalog"], cfg["cataloging_style"],
        cfg.get("catalog_prefix", ""), cfg.get("catalog_suffix", ""))
    cfg["layout_ctx"] = layout_ctx
    cfg["target_catalogs"] = layout_ctx["target_catalogs"]
    log("Source catalog token: %s  ->  base catalog: %s" % (src_token, cfg["catalog"]))
    log("[installer-cataloging-layout FIRED] style=%s target_catalogs=%s" % (
        layout_ctx["style"], layout_ctx["target_catalogs"]))

    plan = OrderedDict((k, []) for k in ("catalog", "schema", "table", "fk", "tag", "metric", "other"))
    for name, ref in schema_files + metric_files:
        rewritten = rewrite_model_sql(raw_by_name[name], src_token, layout_ctx)
        for stmt in filter_catalog_statements(split_sql(rewritten), layout_ctx):
            plan[categorize(stmt)].append(stmt)

    # Merge multiple SET TAGS on the same target+column into one statement -> one
    # metadata commit instead of many (per-table tag commits are the long pole).
    n_tags_raw = len(plan["tag"])
    plan["tag"] = merge_tag_statements(plan["tag"])
    if n_tags_raw != len(plan["tag"]):
        log("Tag merge: %d -> %d statements" % (n_tags_raw, len(plan["tag"])))

    log("Execution plan:")
    total = 0
    for k, v in plan.items():
        if v:
            log("    %-8s %d" % (k, len(v)))
            total += len(v)
    log("    %-8s %d" % ("TOTAL", total))
    return plan


In [ ]:
# === Threaded batch executor with LIVE heartbeat progress + serial retry ===
import threading
import time
from concurrent.futures import ThreadPoolExecutor, as_completed

_IGNORABLE = ("already exists", "already_exists")
# Transient UC/metastore server-side conditions. Under sustained concurrent DDL the
# Unity Catalog API throttles and returns 504/503/429 etc. These are NOT bad SQL -
# they self-heal on a short backoff, so we retry them inline instead of failing.
_TRANSIENT = (
    "uc_client_exception", "failed to contact the unity catalog",
    "gateway time", "504", "503", "502", "429", "too many requests",
    "deadline exceeded", "temporarily unavailable", "request timed out",
    "service unavailable", "connection reset", "concurrent")
PROGRESS_SECONDS = 3   # min seconds between live progress lines (worker-driven)
MAX_RETRIES = 6        # inline attempts per statement for transient UC errors


def _ignorable(msg):
    m = msg.lower()
    return any(tok in m for tok in _IGNORABLE)


def _catalog_exists(catalog):
    """True if the catalog is actually present (defensive check after CREATE)."""
    try:
        spark.sql("DESCRIBE CATALOG `%s`" % catalog)
        return True
    except Exception:
        return False


def _ensure_catalog(catalog, log):
    """Create the catalog robustly across metastore configurations.

    Plain `CREATE CATALOG` works when the metastore has a storage root. On
    workspaces where only Default Storage is enabled (no metastore storage root),
    that statement fails with "Metastore storage root URL does not exist" and
    Default Storage catalogs can ONLY be made in the UI. In that case we fall back
    to a real external location and create the catalog with an explicit MANAGED
    LOCATION under it. Industry-agnostic: external locations are discovered at
    runtime via SHOW EXTERNAL LOCATIONS, never hardcoded."""
    if _catalog_exists(catalog):
        return
    try:
        spark.sql("CREATE CATALOG IF NOT EXISTS `%s`" % catalog)
        if _catalog_exists(catalog):
            return
    except Exception as e:
        low = str(e).lower()
        needs_loc = ("storage root url does not exist" in low
                     or "default storage" in low
                     or "provide a storage location" in low
                     or "managed location" in low)
        if not needs_loc:
            raise
        log("   No metastore storage root on this workspace; discovering an external "
            "location to use as MANAGED LOCATION ...")
    cands = []
    try:
        for r in spark.sql("SHOW EXTERNAL LOCATIONS").collect():
            d = r.asDict()
            name = (d.get("name") or "").strip()
            url = (d.get("url") or "").strip().rstrip("/")
            if url and name and not name.startswith("__"):
                cands.append((name, url))
    except Exception as e:
        log("   Could not list external locations: %s" % str(e)[:160])
    last = None
    for name, url in cands:
        loc = "%s/%s" % (url, catalog)
        try:
            spark.sql("CREATE CATALOG IF NOT EXISTS `%s` MANAGED LOCATION '%s'"
                      % (catalog, loc))
            if _catalog_exists(catalog):
                log("   created catalog `%s` (MANAGED LOCATION under external location `%s`)"
                    % (catalog, name))
                return
        except Exception as e:
            last = str(e)
            continue
    raise Exception(
        "catalog `%s` could not be created. This workspace has no metastore storage "
        "root and Default Storage catalogs are UI-only. Grant an external location "
        "with CREATE MANAGED STORAGE (or create the catalog in the UI), then retry. "
        "Tried %d external location(s). Last error: %s"
        % (catalog, len(cands), (last or "no usable external location found")[:240]))


def _transient(msg):
    m = msg.lower()
    return any(tok in m for tok in _TRANSIENT)


# Author-defect errors in the SOURCE SQL (bad column / function arg / syntax /
# parameter). These are DETERMINISTIC: they fail identically on every attempt, so
# re-running them in retry_failed is pure wasted time (each metric-view recompile
# costs ~50s server-side). We detect them and skip the retry for those statements
# while still reporting them. UNRESOLVED_RELATION is deliberately EXCLUDED - a
# missing table can be a transient ordering artifact that a serial retry fixes.
_DETERMINISTIC = (
    "unresolved_column", "unresolved_routine", "unresolved_field",
    "unresolved_using_column", "parse_syntax_error", "invalid_parameter_value",
    "datatype_mismatch", "ambiguous_reference", "invalid_column",
    "cannot_resolve", "wrong_num_args", "unsupported_call",
)


def _deterministic(msg):
    m = (msg or "").lower()
    return any(tok in m for tok in _DETERMINISTIC)


def _exec_with_backoff(stmt):
    """Run one statement, retrying transient UC errors with exponential backoff +
    jitter. Returns (ok, err). Non-transient errors fail immediately."""
    import random
    delay = 0.5
    last = None
    for attempt in range(MAX_RETRIES + 1):
        try:
            spark.sql(stmt)
            return True, None
        except Exception as e:
            last = str(e)
            if _ignorable(last) or not _transient(last) or attempt == MAX_RETRIES:
                return False, last
            time.sleep(delay + random.uniform(0, delay))
            delay = min(delay * 2, 16.0)
    return False, last


def run_phase(name, statements, threads, batch_size, group=False, serial=False):
    """Run one phase across threads with LIVE worker-driven progress.

    Progress is emitted by the WORKER threads (throttled to one line every
    PROGRESS_SECONDS), not by a separate daemon. A daemon heartbeat gets GIL-starved
    while 32 workers hammer spark.sql, so it only fires between phases. Workers hold
    the GIL between gRPC calls, so emitting from there keeps the log genuinely live.
    Returns [(stmt, err)]."""
    if not statements:
        log("[%s] nothing to do" % name)
        return []
    batches = build_batches(statements, batch_size, group_by_target=group)
    total = len(statements)
    workers = 1 if serial else threads
    prog = {"done": 0, "ok": 0, "ignored": 0, "failed": 0, "last": 0.0, "shown_err": False}
    failures = []
    lock = threading.Lock()
    t0 = time.time()
    prog["last"] = t0
    log("[%s] START %d statements in %d batches, %d workers" % (name, total, len(batches), workers))

    def _emit_progress():
        # caller MUST hold lock
        d, ok, ig, fa = prog["done"], prog["ok"], prog["ignored"], prog["failed"]
        pct = (100.0 * d / total) if total else 100.0
        log("[%s] %d/%d (%.0f%%) ok=%d ignored=%d failed=%d  %.0fs" % (
            name, d, total, pct, ok, ig, fa, time.time() - t0))

    def do_batch(batch):
        for stmt in batch:
            ok, err = _exec_with_backoff(stmt)   # plain spark.sql + transient backoff
            with lock:
                prog["done"] += 1
                if ok:
                    prog["ok"] += 1
                elif _ignorable(err):
                    prog["ignored"] += 1
                else:
                    prog["failed"] += 1
                    failures.append((stmt, err))
                    if not prog["shown_err"]:
                        prog["shown_err"] = True
                        log("[%s] first failure: %s" % (name, " ".join(err.split())[:200]))
                now = time.time()
                if now - prog["last"] >= PROGRESS_SECONDS and prog["done"] < total:
                    prog["last"] = now
                    _emit_progress()

    with ThreadPoolExecutor(max_workers=workers) as ex:
        for fut in as_completed([ex.submit(do_batch, b) for b in batches]):
            fut.result()

    log("[%s] DONE in %.1fs  ok=%d ignored=%d failed=%d" % (
        name, time.time() - t0, prog["ok"], prog["ignored"], prog["failed"]))
    return failures


def retry_failed(failures, passes=3):
    """Re-run failed statements serially; transient/ordering errors self-heal.

    Statements whose first-pass error is a deterministic SOURCE defect (bad column,
    bad function arg, syntax/param error) can NEVER succeed on retry, so we skip
    re-executing them entirely (saves ~50s each - a metric-view recompile cost) and
    keep them as failures. Only transient/ordering errors are retried."""
    deterministic = [(ph, st, er) for (ph, st, er) in failures if _deterministic(er)]
    remaining = [(ph, st, er) for (ph, st, er) in failures if not _deterministic(er)]
    if deterministic:
        log("Skipping retry for %d deterministic source defect(s) - they cannot "
            "self-heal (saved ~%ds)" % (len(deterministic), 50 * len(deterministic)))
    for p in range(1, passes + 1):
        if not remaining:
            break
        log("=== retry pass %d: %d statements ===" % (p, len(remaining)))
        still = []
        for phase, stmt, _ in remaining:
            ok, err = _exec_with_backoff(stmt)
            if ok or _ignorable(err or ""):
                continue
            still.append((phase, stmt, err))
        log("    after pass %d: %d remaining" % (p, len(still)))
        if len(still) == len(remaining):
            remaining = still
            break
        remaining = still
    return deterministic + remaining


In [ ]:
# === Sample data generation (self-contained; reads the installed catalog) ===
"""Generate realistic sample rows for a model that has just been installed.

Source of truth is the INSTALLED CATALOG, not a model file: columns, primary keys
and foreign keys are read back from `information_schema`, which is what the install
phases just wrote. Nothing here depends on the modelling agent.

Referential integrity holds by construction rather than by repair:

    pass 1  every table's primary-key values are generated first, each table in its own
            disjoint value block. A table whose own key contains a foreign key (an order
            line keyed by order_id, line_no) borrows that part from its parent, so pass 1
            visits parents first; ordinary foreign keys impose no order.
    pass 2  every other column is filled; a foreign-key column draws from the pool of
            keys pass 1 already produced for its parent table
    pass 3  the in-memory rows are asserted (unique keys, every foreign key present in
            its parent's key pool, no null in a NOT NULL column) and only then written

Because keys exist before references are filled, an ordinary foreign-key cycle between
two tables is not a special case.

Values come from a deterministic, seeded, name-and-type aware generator. When an LLM
endpoint is reachable it is asked once per table for domain-realistic pools for the
free-text columns; any failure silently falls back to the deterministic generator, so
the LLM can improve realism but can never break a run or its reproducibility.
"""
import datetime
import decimal
import json
import random
import re
import threading
import time
from concurrent.futures import ThreadPoolExecutor, as_completed, wait

SAMPLE_ROW_CHOICES = ["5", "10", "20", "50", "100"]
SAMPLE_INTERNAL_SCHEMAS = ("information_schema", "_metrics", "_install", "_metamodel", "default")
SAMPLE_SEED = 20260801
SAMPLE_MAX_LLM_COLUMNS = 12
SAMPLE_LLM_TIMEOUT_S = 90        # per-table budget for the optional realism pass
SAMPLE_LLM_MAX_ENDPOINT_ERRORS = 3   # transient failures tolerated before giving up


# --------------------------------------------------------------------------------------
# resolved configuration
# --------------------------------------------------------------------------------------

def resolve_sample_config(wget):
    """Read the sample widgets/params through the installer's safe getter."""
    enabled = str(wget("generate_samples", "No")).strip().lower() in ("yes", "true", "1")
    raw_rows = str(wget("sample_rows", "10")).strip()
    rows = int(raw_rows) if raw_rows.isdigit() and int(raw_rows) > 0 else 10
    return {
        "enabled": enabled,
        "rows": rows,
        "seed": int(str(wget("sample_seed", str(SAMPLE_SEED))).strip() or SAMPLE_SEED),
        "threads": max(1, int(str(wget("sample_threads", "8")).strip() or 8)),
        "llm": str(wget("sample_llm", "true")).strip().lower() == "true",
        "llm_endpoints": [e.strip() for e in str(
            wget("sample_llm_endpoints",
                 "databricks-gpt-oss-120b,databricks-meta-llama-3-3-70b-instruct")
        ).split(",") if e.strip()],
    }


# --------------------------------------------------------------------------------------
# the installed model, read back from information_schema
# --------------------------------------------------------------------------------------

class SampleEntity(object):
    """One physical table plus the key metadata needed to populate it."""

    __slots__ = ("catalog", "schema", "table", "columns", "pk", "fks", "keys", "rows")

    def __init__(self, catalog, schema, table):
        self.catalog = catalog
        self.schema = schema
        self.table = table
        self.columns = []      # [{name, type, nullable, position}] in ordinal order
        self.pk = []           # primary-key column names, in key order
        self.fks = []          # [{columns: [...], parent: fqn, parent_columns: [...]}]
        self.keys = []         # pass-1 key tuples, one per row
        self.rows = []         # pass-2 assembled rows

    @property
    def fqn(self):
        return "%s.%s.%s" % (self.catalog, self.schema, self.table)

    @property
    def quoted(self):
        return "`%s`.`%s`.`%s`" % (self.catalog, self.schema, self.table)

    def column(self, name):
        for c in self.columns:
            if c["name"] == name:
                return c
        return None

    def fk_for_column(self, name):
        for fk in self.fks:
            if name in fk["columns"]:
                return fk
        return None


def _rows_of(result):
    """Spark Rows -> plain tuples, so the readers work against any Row implementation."""
    return [tuple(r) for r in result.collect()]


def _sample_read_catalog(spark, catalog, entities):
    """Add every base table of one catalog (columns + PK + FK) into `entities`."""
    skip = ", ".join("'%s'" % s for s in SAMPLE_INTERNAL_SCHEMAS)
    col_rows = _rows_of(spark.sql("""
        SELECT c.table_schema, c.table_name, c.column_name, c.full_data_type,
               c.is_nullable, c.ordinal_position
        FROM `%s`.information_schema.columns c
        JOIN `%s`.information_schema.tables t
          ON t.table_schema = c.table_schema AND t.table_name = c.table_name
        WHERE c.table_schema NOT IN (%s) AND t.table_type <> 'VIEW'
        ORDER BY c.table_schema, c.table_name, c.ordinal_position
    """ % (catalog, catalog, skip)))
    for schema, table, column, dtype, nullable, position in col_rows:
        key = "%s.%s.%s" % (catalog, schema, table)
        ent = entities.get(key)
        if ent is None:
            ent = entities[key] = SampleEntity(catalog, schema, table)
        ent.columns.append({
            "name": column,
            "type": (dtype or "STRING").strip(),
            "nullable": str(nullable).upper() in ("YES", "TRUE"),
            "position": int(position or 0),
        })

    pk_rows = _rows_of(spark.sql("""
        SELECT k.table_schema, k.table_name, k.column_name, k.ordinal_position
        FROM `%s`.information_schema.table_constraints tc
        JOIN `%s`.information_schema.key_column_usage k
          ON tc.constraint_schema = k.constraint_schema
         AND tc.constraint_name = k.constraint_name
        WHERE tc.constraint_type = 'PRIMARY KEY'
        ORDER BY k.table_schema, k.table_name, k.ordinal_position
    """ % (catalog, catalog)))
    for schema, table, column, _pos in pk_rows:
        ent = entities.get("%s.%s.%s" % (catalog, schema, table))
        if ent is not None and column not in ent.pk:
            ent.pk.append(column)

    # referential_constraints maps a foreign key to the parent's UNIQUE/PRIMARY KEY
    # constraint, and key_column_usage lists the ordered columns of either side, so
    # matching on ordinal_position pairs child column to parent column.
    #
    # constraint_column_usage is deliberately NOT used: its constraint_schema is the
    # REFERENCED table's schema, not the schema owning the foreign key, so correlating
    # the two silently loses every cross-schema foreign key (338 of 506 on the
    # restaurants model, which then wrote unresolvable references).
    fk_rows = _rows_of(spark.sql("""
        SELECT rc.constraint_schema, rc.constraint_name,
               ck.table_schema, ck.table_name, ck.column_name, ck.ordinal_position,
               pk.table_catalog, pk.table_schema, pk.table_name, pk.column_name
        FROM `%s`.information_schema.referential_constraints rc
        JOIN `%s`.information_schema.key_column_usage ck
          ON ck.constraint_catalog = rc.constraint_catalog
         AND ck.constraint_schema = rc.constraint_schema
         AND ck.constraint_name = rc.constraint_name
        JOIN `%s`.information_schema.key_column_usage pk
          ON pk.constraint_catalog = rc.unique_constraint_catalog
         AND pk.constraint_schema = rc.unique_constraint_schema
         AND pk.constraint_name = rc.unique_constraint_name
         AND pk.ordinal_position = ck.ordinal_position
        ORDER BY rc.constraint_schema, rc.constraint_name, ck.ordinal_position
    """ % (catalog, catalog, catalog)))
    grouped = {}
    for (cschema, cname, schema, table, column, _pos,
         pcat, pschema, ptable, pcolumn) in fk_rows:
        slot = grouped.setdefault(
            (cschema, cname), {"columns": [], "parent_columns": [], "child": (schema, table),
                               "parent": "%s.%s.%s" % (pcat, pschema, ptable)})
        slot["columns"].append(column)
        slot["parent_columns"].append(pcolumn)
    for slot in grouped.values():
        schema, table = slot.pop("child")
        ent = entities.get("%s.%s.%s" % (catalog, schema, table))
        if ent is not None:
            ent.fks.append(slot)


def read_installed_model(spark, catalogs, log=None):
    """Read every installed base table across the target catalogs."""
    entities = {}
    for catalog in catalogs:
        try:
            _sample_read_catalog(spark, catalog, entities)
        except Exception as err:
            if log:
                log("  sample: could not read catalog `%s` (%s)" % (catalog, str(err)[:160]))
    for ent in entities.values():
        ent.columns.sort(key=lambda c: c["position"])
    if log:
        n_fk = sum(len(e.fks) for e in entities.values())
        n_pk = sum(1 for e in entities.values() if e.pk)
        log("  sample: %d table(s), %d with a primary key, %d foreign key(s)"
            % (len(entities), n_pk, n_fk))
    return entities


# --------------------------------------------------------------------------------------
# type helpers
# --------------------------------------------------------------------------------------

_DECIMAL_RE = re.compile(r"(?:DECIMAL|NUMERIC)\s*\(\s*(\d+)\s*,\s*(\d+)\s*\)", re.IGNORECASE)
_INT_CEILINGS = (("TINYINT", 127), ("SMALLINT", 32767), ("BYTE", 127), ("SHORT", 32767))


def _decimal_precision(dtype):
    m = _DECIMAL_RE.search(dtype or "")
    if m:
        return int(m.group(1)), int(m.group(2))
    return 38, 0 if "DECIMAL" not in (dtype or "").upper() else 2


def type_ceiling(dtype):
    """Largest magnitude the declared type holds, or None when it cannot overflow.

    Without this a DECIMAL(5,4) column is sampled from a plausible business range,
    clamped on write and every row lands on 9.9999.
    """
    upper = (dtype or "").upper()
    if "DECIMAL" in upper or "NUMERIC" in upper:
        precision, scale = _decimal_precision(dtype)
        return float(10 ** (precision - scale)) - (float(10 ** -scale) if scale else 1.0)
    for token, ceiling in _INT_CEILINGS:
        if upper.startswith(token):
            return float(ceiling)
    if upper.startswith("INT"):
        return 2147483647.0
    return None


def type_family(dtype):
    upper = (dtype or "STRING").upper()
    if upper.startswith(("ARRAY", "MAP", "STRUCT")):
        return "complex"
    if upper.startswith("BINARY"):
        return "binary"
    if upper.startswith("BOOLEAN"):
        return "boolean"
    if upper.startswith("TIMESTAMP"):
        return "timestamp"
    if upper.startswith("DATE"):
        return "date"
    if upper.startswith(("DECIMAL", "NUMERIC")):
        return "decimal"
    if upper.startswith(("DOUBLE", "FLOAT", "REAL")):
        return "float"
    if upper.startswith(("INT", "BIGINT", "SMALLINT", "TINYINT", "LONG", "SHORT", "BYTE")):
        return "integer"
    return "string"


def _coerce(value, dtype):
    """Return `value` as the exact Python type Spark expects for `dtype`."""
    family = type_family(dtype)
    if value is None:
        return None
    if family == "decimal":
        precision, scale = _decimal_precision(dtype)
        quant = decimal.Decimal(1).scaleb(-scale)
        coerced = decimal.Decimal(str(value)).quantize(quant, rounding=decimal.ROUND_HALF_UP)
        # A DECIMAL(p,s) holds at most (p - s) integer digits. LLM value pools do not pass
        # through numeric_range's type_ceiling clamp, so an out-of-range magnitude (e.g.
        # 1234567.89 for DECIMAL(6,2)) reaches here and Spark rejects the whole write on
        # decimal-precision overflow, which skips every table that references it (live:
        # coffee_roastery wholesale.sales_rep failed, 7 dependents emptied). Clamp the
        # magnitude to what the declared precision holds so the value always fits.
        limit = decimal.Decimal(10) ** (precision - scale) - quant
        if coerced > limit:
            coerced = limit
        elif coerced < -limit:
            coerced = -limit
        return coerced
    if family == "integer":
        return int(value)
    if family == "float":
        return float(value)
    if family == "boolean":
        return bool(value)
    if family == "string":
        return value if isinstance(value, str) else str(value)
    return value


# --------------------------------------------------------------------------------------
# deterministic, name-aware value pools
# --------------------------------------------------------------------------------------

_FIRST_NAMES = ["Amara", "Liam", "Sofia", "Noah", "Yuki", "Mateo", "Aisha", "Ethan", "Priya",
                "Lucas", "Nadia", "Omar", "Elena", "Kai", "Zara", "Hugo", "Mei", "Idris",
                "Freya", "Diego", "Layla", "Anton", "Chiara", "Rafael"]
_LAST_NAMES = ["Okafor", "Nguyen", "Rossi", "Haddad", "Silva", "Kowalski", "Tanaka", "Mbeki",
               "Andersen", "Novak", "Fernandez", "Bakker", "Costa", "Petrov", "Sharma",
               "Dubois", "Larsen", "Moreau", "Ibrahim", "Weber"]
_CITIES = ["Amsterdam", "Nairobi", "Osaka", "Lisbon", "Toronto", "Dubai", "Santiago", "Oslo",
           "Cape Town", "Seoul", "Munich", "Melbourne", "Kraków", "Bogotá", "Helsinki",
           "Casablanca", "Auckland", "Bengaluru", "Montréal", "Valencia"]
_STREETS = ["Harbour Way", "Cedar Lane", "Market Street", "Willow Road", "Station Approach",
            "Granite Avenue", "Old Mill Road", "Riverside Walk", "Foundry Street",
            "Kingfisher Close"]
_COUNTRIES = ["Netherlands", "Kenya", "Japan", "Portugal", "Canada", "United Arab Emirates",
              "Chile", "Norway", "South Africa", "South Korea", "Germany", "Australia"]
_COUNTRY_CODES = ["NL", "KE", "JP", "PT", "CA", "AE", "CL", "NO", "ZA", "KR", "DE", "AU"]
_CURRENCY_CODES = ["USD", "EUR", "GBP", "JPY", "AED", "ZAR", "CAD", "AUD", "CHF", "SGD"]
_LANGUAGE_CODES = ["en", "fr", "de", "es", "pt", "ar", "ja", "ko", "nl", "sw"]
_UNITS = ["kg", "g", "lb", "litre", "each", "case", "pallet", "metre", "hour"]
_DOMAIN_WORDS = ["northwind", "brightline", "harborstone", "cedarpoint", "vantage",
                 "meridian", "solstice", "clearwater"]

_CATEGORICAL_POOLS = (
    (("status",), ["active", "pending", "completed", "cancelled", "on_hold", "closed"]),
    (("state",), ["active", "inactive", "suspended", "archived"]),
    (("stage",), ["intake", "qualification", "execution", "review", "closed"]),
    (("priority",), ["low", "medium", "high", "critical"]),
    (("severity",), ["info", "minor", "major", "critical"]),
    (("tier", "grade", "class", "band"), ["standard", "premium", "enterprise", "basic"]),
    (("channel",), ["web", "mobile", "branch", "partner", "call_center", "field"]),
    (("method",), ["card", "bank_transfer", "cash", "wallet", "direct_debit"]),
    (("frequency", "cadence"), ["daily", "weekly", "monthly", "quarterly", "annual"]),
    (("currency",), _CURRENCY_CODES),
    (("country", "nationality"), _COUNTRIES),
    (("language", "locale"), _LANGUAGE_CODES),
    (("city", "town"), _CITIES),
    (("region", "zone", "area", "territory"),
     ["north", "south", "east", "west", "central", "emea", "apac", "amer"]),
    (("unit", "uom"), _UNITS),
    (("category", "type", "kind", "segment"),
     ["standard", "express", "bulk", "custom", "seasonal", "recurring"]),
    (("direction",), ["inbound", "outbound", "internal"]),
    (("source", "origin"), ["manual", "import", "api", "batch", "partner_feed"]),
    (("gender", "sex"), ["female", "male", "unspecified"]),
)
_CODE_STEM_POOLS = {"country": _COUNTRY_CODES, "currency": _CURRENCY_CODES,
                    "language": _LANGUAGE_CODES, "locale": _LANGUAGE_CODES}

# (lower token, upper token) -> the lower one must not be later than the upper one.
_TEMPORAL_ORDER_TOKENS = (
    ("start", "end"), ("begin", "end"), ("from", "to"), ("open", "close"),
    ("created", "updated"), ("created", "modified"), ("created", "closed"),
    ("issued", "expiry"), ("issue", "expiry"), ("issue", "due"), ("effective", "expiry"),
    ("valid", "expiry"), ("order", "ship"), ("order", "delivery"), ("ship", "delivery"),
    ("entry", "exit"), ("arrival", "departure"), ("admission", "discharge"),
    ("hire", "termination"), ("first", "last"), ("request", "approval"),
    ("approval", "completion"), ("booking", "checkin"), ("checkin", "checkout"),
)
# token -> (low bound, high bound, decimal places)
_NUMERIC_RANGES = (
    (("pct", "percent", "percentage", "rate", "ratio", "utilization", "margin"), 0.0, 100.0, 2),
    (("score", "rating", "index"), 0.0, 100.0, 1),
    (("latitude",), -90.0, 90.0, 6),
    (("longitude",), -180.0, 180.0, 6),
    (("temperature", "temp"), -20.0, 45.0, 1),
    (("weight", "mass"), 0.1, 2500.0, 2),
    (("height", "width", "length", "depth", "distance"), 0.1, 500.0, 2),
    (("volume", "capacity"), 1.0, 10000.0, 2),
    (("amount", "total", "value", "revenue", "cost", "price", "balance", "fee",
      "charge", "salary", "budget"), 5.0, 250000.0, 2),
    (("discount", "tax", "vat"), 0.0, 2500.0, 2),
    (("quantity", "qty", "count", "units", "items"), 1.0, 500.0, 0),
    (("age", "years"), 18.0, 85.0, 0),
    (("duration", "minutes", "elapsed"), 1.0, 480.0, 0),
    (("seconds",), 1.0, 3600.0, 0),
    (("hours",), 1.0, 24.0, 1),
    (("days",), 1.0, 365.0, 0),
    (("year",), 2015.0, 2026.0, 0),
    (("sequence", "order", "position", "rank", "step", "level"), 1.0, 20.0, 0),
    (("version",), 1.0, 9.0, 0),
)
_TEXT_TOKENS = ("description", "comment", "notes", "note", "remark", "summary",
                "justification", "reason", "detail", "instruction", "message")
_PERSON_TOKENS = ("person", "customer", "employee", "contact", "member", "patient",
                  "passenger", "student", "user", "owner", "manager", "agent", "driver",
                  "supplier", "vendor", "author", "guest", "client")


def tokens_of(column_name):
    return [t for t in re.split(r"[^a-z0-9]+", str(column_name).lower()) if t]


def categorical_pool(parts):
    """(pool, matched_via_code_suffix) for a column's tokens, else (None, False).

    Matching only the last token leaves `country_code` / `currency_code` unmatched,
    which is the commonest shape a generated model produces, so a trailing
    `code`/`cd` defers to the token before it. `country_name` stays a name.
    """
    if not parts:
        return None, False
    last = parts[-1]
    via_code = last in ("code", "cd") and len(parts) > 1
    if via_code:
        stem = parts[-2]
        if stem in _CODE_STEM_POOLS:
            return _CODE_STEM_POOLS[stem], True
    key = parts[-2] if via_code else last
    for tokens, pool in _CATEGORICAL_POOLS:
        if key in tokens:
            return pool, via_code
    for part in reversed(parts):
        for tokens, pool in _CATEGORICAL_POOLS:
            if part in tokens:
                return pool, False
    return None, False


def numeric_range(column_name, dtype):
    """(low, high, decimals) for a column, clamped to what the declared type holds."""
    parts = tokens_of(column_name)
    low, high, places = None, None, None
    for tokens, lo, hi, dp in _NUMERIC_RANGES:
        if any(p in tokens for p in parts):
            low, high, places = lo, hi, dp
            break
    if low is None:
        family = type_family(dtype)
        low, high, places = (1.0, 1000.0, 0) if family == "integer" else (1.0, 10000.0, 2)
    if type_family(dtype) == "integer":
        places = 0
    if type_family(dtype) == "decimal":
        places = min(places, _decimal_precision(dtype)[1])
    ceiling = type_ceiling(dtype)
    if ceiling is not None and high > ceiling:
        high = ceiling
        if low > high:
            low = 0.0 if high >= 0 else high
    return low, high, places


# --------------------------------------------------------------------------------------
# temporal coherence
# --------------------------------------------------------------------------------------

def _token_position(parts, token):
    for i, part in enumerate(parts):
        if part == token or (len(token) >= 4 and part.startswith(token)):
            return i
    return -1


def temporal_edges(names):
    """Directed (earlier, later) pairs implied by the column names."""
    edges = []
    for lower_token, upper_token in _TEMPORAL_ORDER_TOKENS:
        lows = [n for n in names if _token_position(tokens_of(n), lower_token) >= 0]
        highs = [n for n in names if _token_position(tokens_of(n), upper_token) >= 0]
        for low in lows:
            low_parts = tokens_of(low)
            if _token_position(low_parts, upper_token) >= 0:
                # the name carries both tokens; the later one wins its role
                if _token_position(low_parts, upper_token) > _token_position(low_parts, lower_token):
                    continue
            for high in highs:
                if high == low:
                    continue
                high_parts = tokens_of(high)
                if (_token_position(high_parts, lower_token) >= 0
                        and _token_position(high_parts, lower_token)
                        > _token_position(high_parts, upper_token)):
                    continue
                if (low, high) not in edges:
                    edges.append((low, high))
    return edges


def temporal_order_plan(names):
    """[(column, [columns it must not precede])] in an order where repairs stick.

    A repair pushes a column forward past its predecessors, so the predecessors have
    to be final first: the plan is a topological order over the name-implied edges.
    Columns in a naming cycle are dropped rather than repaired arbitrarily.
    """
    edges = temporal_edges(names)
    parents = dict((n, []) for n in names)
    for low, high in edges:
        parents[high].append(low)
    resolved, plan = set(), []
    remaining = [n for n in names]
    while remaining:
        ready = [n for n in remaining if all(p in resolved for p in parents[n])]
        if not ready:
            break
        for name in ready:
            if parents[name]:
                plan.append((name, list(parents[name])))
            resolved.add(name)
            remaining.remove(name)
    return plan


def enforce_temporal_order(row, column_types, rnd):
    """Push any date/timestamp in `row` that precedes a predecessor forward."""
    temporal = [n for n, t in column_types.items()
                if type_family(t) in ("date", "timestamp") and row.get(n) is not None]
    repaired = 0
    for name, predecessors in temporal_order_plan(temporal):
        value = row.get(name)
        if not isinstance(value, (datetime.date, datetime.datetime)):
            continue
        as_dt = value if isinstance(value, datetime.datetime) else \
            datetime.datetime.combine(value, datetime.time())
        floor = None
        for predecessor in predecessors:
            other = row.get(predecessor)
            if not isinstance(other, (datetime.date, datetime.datetime)):
                continue
            other_dt = other if isinstance(other, datetime.datetime) else \
                datetime.datetime.combine(other, datetime.time())
            if floor is None or other_dt > floor:
                floor = other_dt
        if floor is None:
            continue
        # A date may legitimately land on the same day as a timestamp predecessor, so
        # only a strictly earlier DAY counts as out of order for a date column.
        if not isinstance(value, datetime.datetime):
            if as_dt.date() >= floor.date():
                continue
        elif as_dt >= floor:
            continue
        moved = floor + datetime.timedelta(days=rnd.randint(1, 240), hours=rnd.randint(0, 23))
        row[name] = moved.date() if not isinstance(value, datetime.datetime) else moved
        repaired += 1
    return repaired


# --------------------------------------------------------------------------------------
# value generation
# --------------------------------------------------------------------------------------

def _slug(text, length=3):
    letters = re.sub(r"[^A-Z]", "", str(text).upper())
    return (letters + "XXX")[:length]


def _key_block(entity):
    """A disjoint numeric block per table so keys never collide across tables."""
    digest = 0
    for ch in entity.fqn:
        digest = (digest * 131 + ord(ch)) & 0x7FFFFFFF
    return 100000 + (digest % 8999) * 100000


def identifying_fks(entity):
    """Foreign keys whose columns are part of this table's own primary key.

    An order line keyed by (order_id, line_no) OWNS its parent's key, so that column
    cannot be minted from this table's block: it has to be a real parent key or the
    child references a row that does not exist.
    """
    if not entity.pk:
        return []
    return [fk for fk in entity.fks if any(c in entity.pk for c in fk["columns"])]


def key_generation_order(entities):
    """Table order for pass 1: a table that borrows a key comes after its parent.

    Only identifying foreign keys constrain the order, so an ordinary foreign-key
    cycle is unaffected. A cycle of identifying keys cannot be satisfied at all, so
    those tables are emitted last and fall back to their own key block.
    """
    parents = {}
    for fqn, entity in entities.items():
        needed = set()
        for fk in identifying_fks(entity):
            if fk["parent"] in entities and fk["parent"] != fqn:
                needed.add(fk["parent"])
        parents[fqn] = needed
    ordered, placed = [], set()
    remaining = list(entities.keys())
    while remaining:
        ready = [f for f in remaining if parents[f] <= placed]
        if not ready:
            ordered.extend(sorted(remaining))
            break
        for fqn in sorted(ready):
            ordered.append(fqn)
            placed.add(fqn)
            remaining.remove(fqn)
    return ordered


def _strongly_connected(nodes, parents):
    """Groups of tables that reach each other through foreign keys, Tarjan, iterative.

    Iterative because a wide model can nest deeper than the interpreter's recursion limit.
    """
    index, low, on_stack, stack, order = {}, {}, set(), [], []
    groups, counter = [], [0]
    for root in nodes:
        if root in index:
            continue
        work = [(root, iter(sorted(parents[root])))]
        index[root] = low[root] = counter[0]
        counter[0] += 1
        stack.append(root)
        on_stack.add(root)
        while work:
            node, children = work[-1]
            advanced = False
            for child in children:
                if child not in index:
                    index[child] = low[child] = counter[0]
                    counter[0] += 1
                    stack.append(child)
                    on_stack.add(child)
                    work.append((child, iter(sorted(parents[child]))))
                    advanced = True
                    break
                if child in on_stack:
                    low[node] = min(low[node], index[child])
            if advanced:
                continue
            work.pop()
            if work:
                low[work[-1][0]] = min(low[work[-1][0]], low[node])
            if low[node] == index[node]:
                group = []
                while True:
                    member = stack.pop()
                    on_stack.discard(member)
                    group.append(member)
                    if member == node:
                        break
                groups.append(frozenset(group))
    del order
    return groups


def _downstream_of(entities, roots):
    """Every table that reaches `roots` through foreign keys, plus the roots themselves.

    A table whose parent never got rows must not be written, and neither must ITS
    children, so the closure has to be transitive rather than one level deep.
    """
    children = {}
    for fqn, entity in entities.items():
        for fk in entity.fks:
            if fk["parent"] in entities and fk["parent"] != fqn:
                children.setdefault(fk["parent"], set()).add(fqn)
    reached, queue = set(roots), list(roots)
    while queue:
        for child in children.get(queue.pop(), ()):
            if child not in reached:
                reached.add(child)
                queue.append(child)
    return reached


def write_order(entities):
    """Waves of tables to write, parents before children, over EVERY foreign key.

    Key generation only needs identifying keys ordered, but WRITING needs all of them:
    if a child lands and its parent's insert then fails, the child's references point at
    rows that will never exist. Writing parents first means an aborted pass leaves later
    tables empty, and an empty child cannot be an orphan.

    Tables inside a foreign-key cycle cannot be layered against each other, so they share
    one wave. Condensing each cycle to a single node first keeps that concession to the
    cycle itself: everything downstream of a cycle still gets its own later wave.
    """
    parents = {}
    for fqn, entity in entities.items():
        parents[fqn] = set(fk["parent"] for fk in entity.fks
                           if fk["parent"] in entities and fk["parent"] != fqn)

    groups = _strongly_connected(sorted(entities), parents)
    group_of = dict((member, i) for i, g in enumerate(groups) for member in g)
    group_deps = {}
    for i, group in enumerate(groups):
        group_deps[i] = set(group_of[p] for member in group for p in parents[member]
                            if group_of[p] != i)

    waves, placed = [], set()
    remaining = set(range(len(groups)))
    while remaining:
        ready = sorted(i for i in remaining if group_deps[i] <= placed)
        if not ready:                       # unreachable: the condensation is acyclic
            ready = sorted(remaining)
        wave = sorted(member for i in ready for member in groups[i])
        waves.append(wave)
        placed.update(ready)
        remaining.difference_update(ready)
    return waves


def _borrowed_key_columns(entity, entities):
    """{pk column: (parent entity, position in the parent key)} for identifying keys."""
    borrowed = {}
    for fk in identifying_fks(entity):
        parent = entities.get(fk["parent"]) if entities else None
        if parent is None or parent.fqn == entity.fqn or not parent.keys or not parent.pk:
            continue
        for position, column in enumerate(fk["columns"]):
            if column not in entity.pk:
                continue
            parent_column = fk["parent_columns"][position] \
                if position < len(fk["parent_columns"]) else fk["parent_columns"][0]
            parent_position = parent.pk.index(parent_column) \
                if parent_column in parent.pk else 0
            borrowed[column] = (parent, parent_position)
    return borrowed


def _generate_borrowed_keys(entity, rows, seed, borrowed):
    """Keys for a table whose primary key contains a parent's key."""
    rnd = random.Random("%s|borrowed|%s" % (entity.fqn, seed))
    parents = sorted(set(p.fqn for p, _ in borrowed.values()))
    lead = borrowed[next(c for c in entity.pk if c in borrowed)][0]
    free = [c for c in entity.pk if c not in borrowed]
    if not free:
        # The key IS the parent's key, so this is a 1:1 extension: one row per parent
        # row at most, otherwise the key could not stay unique.
        rows = min(rows, len(lead.keys))
        picks = list(range(rows))
    else:
        picks = _fk_parent_indices(rows, len(lead.keys), rnd)
    used, keys = {}, []
    for index in range(rows):
        pick = picks[index]
        values = []
        for column in entity.pk:
            if column in borrowed:
                parent, position = borrowed[column]
                source = parent.keys[pick % len(parent.keys)]
                values.append(source[position] if position < len(source) else source[0])
            else:
                values.append(None)
        stem = tuple(v for v in values if v is not None)
        counter = used.get(stem, 0) + 1
        used[stem] = counter
        for slot, column in enumerate(entity.pk):
            if values[slot] is not None:
                continue
            column_type = (entity.column(column) or {"type": "INT"})["type"]
            family = type_family(column_type)
            if family in ("integer", "decimal", "float"):
                values[slot] = _coerce(counter, column_type)
            elif family in ("date", "timestamp"):
                moment = datetime.datetime(2025, 1, 1) + datetime.timedelta(days=counter)
                values[slot] = moment.date() if family == "date" else moment
            else:
                values[slot] = "%s-%03d" % (_slug(column, 3), counter)
        keys.append(tuple(values))
    _ = parents
    entity.keys = keys
    return keys


def _key_part(ordinal, dtype, prefix, width=6, day=None):
    """One key component, in the type its column declares.

    Every key part is minted here so none can carry a type the column cannot
    store: a string in a DATE or DECIMAL key column makes Spark reject the whole
    table on write, which surfaces as a failed install rather than as bad data.
    """
    family = type_family(dtype)
    if family in ("integer", "decimal", "float"):
        ceiling = type_ceiling(dtype)
        if ceiling is not None and ordinal > ceiling:
            # A SMALLINT or DECIMAL(5,0) key column cannot hold this table's key
            # block, and an out-of-range value fails the write for the whole table.
            # Fold into range instead; the integrity gate reports any duplicates.
            ordinal = ordinal % (int(ceiling) or 1)
        return _coerce(ordinal, dtype)
    if family in ("date", "timestamp"):
        moment = datetime.datetime(2025, 1, 1) + datetime.timedelta(
            days=(ordinal if day is None else day) % 36500)
        return moment.date() if family == "date" else moment
    if family == "boolean":
        return bool(ordinal % 2)
    return "%s-%0*d" % (prefix, width, ordinal)


def generate_keys(entity, rows, seed, entities=None):
    """Pass 1: one unique key tuple per row, typed to the declared key columns."""
    if not entity.pk:
        entity.keys = [tuple() for _ in range(rows)]
        return entity.keys
    borrowed = _borrowed_key_columns(entity, entities) if entities else {}
    if borrowed:
        return _generate_borrowed_keys(entity, rows, seed, borrowed)
    base = _key_block(entity)
    prefix = "%s%s" % (_slug(entity.schema, 2), _slug(entity.table, 3))
    keys = []
    for index in range(rows):
        ordinal = base + index
        tuple_values = []
        for depth, column_name in enumerate(entity.pk):
            column = entity.column(column_name) or {"type": "STRING"}
            # A composite key stays unique because only the LAST column carries the
            # row ordinal; the earlier ones repeat in blocks, as real keys do.
            if depth < len(entity.pk) - 1:
                part = index // max(1, (depth + 1) * 2) + 1
                tuple_values.append(_key_part(part, column["type"], prefix, 3))
            else:
                tuple_values.append(
                    _key_part(ordinal, column["type"], prefix, 6, day=index))
        keys.append(tuple(tuple_values))
    if len(entity.pk) > 1:
        # Blocked leading columns can repeat a tuple when a table has very few rows;
        # widen the last part until every tuple is distinct.
        last_type = (entity.column(entity.pk[-1]) or {"type": "STRING"})["type"]
        seen, widened, bump = set(), [], 0
        for index, key in enumerate(keys):
            # Bounded: a domain with fewer distinct values than rows (a BOOLEAN key
            # column, say) cannot be widened, and the integrity gate should report
            # that honestly rather than the loop spinning forever.
            attempts = 0
            while key in seen and attempts < rows + 16:
                bump += 1
                attempts += 1
                key = key[:-1] + (_key_part(base + bump, last_type, prefix, 6,
                                            day=rows + bump),)
            seen.add(key)
            widened.append(key)
        keys = widened
    entity.keys = keys
    return keys


def generate_all_keys(entities, rows, seed):
    """Pass 1 across the model, parents of identifying keys first."""
    for fqn in key_generation_order(entities):
        generate_keys(entities[fqn], rows, seed, entities)
    return entities


def _fk_parent_indices(rows, parent_row_count, rnd):
    """Row -> parent row index, covering every parent once before repeating.

    Straight random choice leaves parents unreferenced, which makes a demo join look
    empty; cycling a shuffled parent list first guarantees fan-out.
    """
    if parent_row_count <= 0:
        return []
    order = list(range(parent_row_count))
    rnd.shuffle(order)
    picks = []
    while len(picks) < rows:
        picks.extend(order[:min(len(order), rows - len(picks))])
        rnd.shuffle(order)
    return picks[:rows]


def generate_value(column_name, dtype, rnd, row_index, pools=None):
    """One deterministic, name-aware value for a non-key column."""
    parts = tokens_of(column_name)
    family = type_family(dtype)
    pool = (pools or {}).get(column_name)
    if pool:
        value = pool[row_index % len(pool)] if len(pool) < 4 else rnd.choice(pool)
        if family == "string":
            return str(value)

    if family == "complex":
        upper = dtype.upper()
        return {} if upper.startswith("MAP") else ([] if upper.startswith("ARRAY") else None)
    if family == "binary":
        return bytes(rnd.getrandbits(8) for _ in range(8))
    if family == "boolean":
        return rnd.random() < 0.5

    if family in ("date", "timestamp"):
        moment = (datetime.datetime(2024, 1, 1)
                  + datetime.timedelta(days=rnd.randint(0, 730),
                                       hours=rnd.randint(0, 23),
                                       minutes=rnd.randint(0, 59),
                                       seconds=rnd.randint(0, 59)))
        return moment.date() if family == "date" else moment

    if family in ("integer", "decimal", "float"):
        if any(p in ("year",) for p in parts):
            return _coerce(rnd.randint(2015, 2026), dtype)
        low, high, places = numeric_range(column_name, dtype)
        raw = rnd.uniform(low, high)
        return _coerce(round(raw, places) if places else int(round(raw)), dtype)

    # strings
    last = parts[-1] if parts else ""
    joined = "_".join(parts)
    if last in ("email",) or "email" in parts:
        return "%s.%s@%s.com" % (rnd.choice(_FIRST_NAMES).lower(),
                                 rnd.choice(_LAST_NAMES).lower(),
                                 rnd.choice(_DOMAIN_WORDS))
    if "phone" in parts or "mobile" in parts or "telephone" in parts:
        return "+%d %d %d" % (rnd.randint(1, 99), rnd.randint(100, 999), rnd.randint(100000, 999999))
    if "url" in parts or "website" in parts or "link" in parts:
        return "https://www.%s.example/%s" % (rnd.choice(_DOMAIN_WORDS), rnd.choice(parts) if parts else "page")
    if last in ("postcode", "postal", "zip", "zipcode") or "postal" in parts:
        return "%d%s" % (rnd.randint(1000, 9999), _slug(rnd.choice(_CITIES), 2))
    if "street" in parts or "address" in joined:
        return "%d %s" % (rnd.randint(1, 240), rnd.choice(_STREETS))
    if "first" in parts and "name" in parts:
        return rnd.choice(_FIRST_NAMES)
    if "last" in parts and "name" in parts or "surname" in parts:
        return rnd.choice(_LAST_NAMES)
    if last == "name" and any(p in _PERSON_TOKENS for p in parts):
        return "%s %s" % (rnd.choice(_FIRST_NAMES), rnd.choice(_LAST_NAMES))

    pool, _via_code = categorical_pool(parts)
    if pool:
        return rnd.choice(pool)

    if any(p in _TEXT_TOKENS for p in parts):
        subject = " ".join(p for p in parts if p not in _TEXT_TOKENS) or "record"
        return "%s %s for reference %s-%04d." % (
            rnd.choice(["Reviewed", "Confirmed", "Logged", "Adjusted", "Verified"]),
            subject.replace("_", " "), _slug(subject, 3), rnd.randint(1, 9999))
    if last == "name" or last == "title" or last == "label":
        stem = " ".join(p for p in parts if p not in ("name", "title", "label")) or "record"
        return "%s %s %d" % (stem.replace("_", " ").title(),
                             rnd.choice(["Alpha", "Beta", "Core", "Prime", "North"]),
                             rnd.randint(1, 99))
    if last in ("code", "cd", "ref", "reference", "number", "no", "sku", "id", "key"):
        stem = parts[-2] if len(parts) > 1 else (parts[0] if parts else "ref")
        return "%s-%06d" % (_slug(stem, 3), rnd.randint(1, 999999))
    stem = joined.replace("_", " ") or "value"
    return "%s %s" % (stem.title(), rnd.randint(100, 9999))


# --------------------------------------------------------------------------------------
# optional LLM realism pass
# --------------------------------------------------------------------------------------

_LLM_STATE = {"broken": set(), "errors": {}, "lock": threading.Lock()}
_LLM_JSON_RE = re.compile(r"\{.*\}", re.DOTALL)


def _llm_candidate_columns(entity):
    """Free-text columns an LLM can make more realistic than the generic generator."""
    keys = set(entity.pk)
    for fk in entity.fks:
        keys.update(fk["columns"])
    out = []
    for column in entity.columns:
        if column["name"] in keys or type_family(column["type"]) != "string":
            continue
        parts = tokens_of(column["name"])
        if parts and parts[-1] in ("id", "key", "code", "cd"):
            continue
        out.append(column["name"])
    return out[:SAMPLE_MAX_LLM_COLUMNS]


def llm_value_pools(spark, entity, columns, endpoints, rows, log=None):
    """Ask one LLM endpoint for realistic value pools. {} on any problem."""
    if not columns or not endpoints:
        return {}
    want = max(rows, 8)
    prompt = (
        "You generate realistic sample data for a data model. Table `%s`.`%s`. "
        "For each column below return a JSON array of %d distinct, realistic values "
        "that a production system would hold. Values must be plain strings, no "
        "placeholders, no numbering, no commentary. Reply with ONLY a JSON object "
        "mapping each column name to its array. Columns: %s"
        % (entity.schema, entity.table, want, ", ".join(columns)))
    literal = prompt.replace("\\", "\\\\").replace("'", "\\'")
    for endpoint in endpoints:
        with _LLM_STATE["lock"]:
            if endpoint in _LLM_STATE["broken"]:
                continue
        try:
            raw = spark.sql("SELECT ai_query('%s', '%s') AS r" % (endpoint, literal)).collect()
            text = raw[0][0] if raw else ""
            match = _LLM_JSON_RE.search(text or "")
            if not match:
                continue
            parsed = json.loads(match.group(0))
            pools = {}
            for column in columns:
                values = parsed.get(column)
                if isinstance(values, list):
                    clean = [str(v) for v in values
                             if isinstance(v, (str, int, float)) and str(v).strip()]
                    if len(clean) >= 3:
                        pools[column] = clean
            if pools:
                return pools
        except Exception as err:
            # One table's call can fail on a transient rate limit while the endpoint is
            # perfectly healthy, so retire it only after repeated failures. Retiring on
            # the first error costs every remaining table its realistic values.
            with _LLM_STATE["lock"]:
                count = _LLM_STATE["errors"].get(endpoint, 0) + 1
                _LLM_STATE["errors"][endpoint] = count
                retired = count >= SAMPLE_LLM_MAX_ENDPOINT_ERRORS
                if retired:
                    _LLM_STATE["broken"].add(endpoint)
            if log and retired:
                log("  sample: LLM endpoint %s retired after %d failures (%s) - "
                    "deterministic values"
                    % (endpoint, count, str(err).split("\n")[0][:120]))
    return {}


# --------------------------------------------------------------------------------------
# row assembly
# --------------------------------------------------------------------------------------

def generate_rows(entity, entities, rows, seed, pools=None):
    """Pass 2: assemble every row, drawing foreign keys from the parents' key pools."""
    rnd = random.Random("%s|rows|%s" % (entity.fqn, seed))
    column_types = dict((c["name"], c["type"]) for c in entity.columns)
    pk_index = dict((name, i) for i, name in enumerate(entity.pk))
    if entity.pk and entity.keys is not None:
        # A 1:1 extension table cannot hold more rows than its parent, so pass 1 is
        # the authority on how many rows this table gets.
        rows = len(entity.keys)

    fk_assignment = {}
    for fk in entity.fks:
        parent = entities.get(fk["parent"])
        fk_rnd = random.Random("%s|fk|%s|%s" % (entity.fqn, ",".join(fk["columns"]), seed))
        if parent is None or not parent.keys or not parent.pk:
            fk_assignment[tuple(fk["columns"])] = None
            continue
        if parent.fqn == entity.fqn:
            # A self reference points at an earlier row so the data has no cycle.
            picks = [None] + [fk_rnd.randint(0, i - 1) for i in range(1, rows)]
        else:
            picks = _fk_parent_indices(rows, len(parent.keys), fk_rnd)
        fk_assignment[tuple(fk["columns"])] = (parent, fk, picks)

    assembled = []
    for index in range(rows):
        row = {}
        for depth, name in enumerate(entity.pk):
            row[name] = entity.keys[index][depth] if index < len(entity.keys) else None
        for key_columns, assignment in fk_assignment.items():
            if assignment is None:
                for name in key_columns:
                    if name not in pk_index:
                        column = entity.column(name) or {"type": "STRING", "nullable": True}
                        row[name] = None if column["nullable"] else generate_value(
                            name, column["type"], rnd, index)
                continue
            parent, fk, picks = assignment
            pick = picks[index] if index < len(picks) else None
            for position, name in enumerate(key_columns):
                if name in pk_index:
                    continue
                column = entity.column(name) or {"type": "STRING", "nullable": True}
                if pick is None:
                    # first row of a self reference: null when allowed, else itself
                    row[name] = None if column["nullable"] else (
                        entity.keys[index][0] if entity.keys and entity.keys[index] else None)
                    continue
                parent_column = fk["parent_columns"][position] \
                    if position < len(fk["parent_columns"]) else fk["parent_columns"][0]
                parent_position = parent.pk.index(parent_column) \
                    if parent_column in parent.pk else 0
                value = parent.keys[pick][parent_position] \
                    if parent_position < len(parent.keys[pick]) else None
                row[name] = _coerce(value, column["type"])
        for column in entity.columns:
            name = column["name"]
            if name in row:
                continue
            row[name] = generate_value(name, column["type"], rnd, index, pools)
        enforce_temporal_order(row, column_types, rnd)
        assembled.append(row)
    entity.rows = assembled
    return assembled


# --------------------------------------------------------------------------------------
# integrity assertions - the gate before anything is written
# --------------------------------------------------------------------------------------

def assert_integrity(entities):
    """Every violation found in the assembled rows, as readable strings."""
    problems = []
    key_pools = {}
    for ent in entities.values():
        if not ent.pk or not ent.rows:
            continue
        pool = set()
        for row in ent.rows:
            pool.add(tuple(row.get(c) for c in ent.pk))
        key_pools[ent.fqn] = pool
        if len(pool) != len(ent.rows):
            problems.append("%s: primary key %s has %d duplicate row(s)"
                            % (ent.fqn, "+".join(ent.pk), len(ent.rows) - len(pool)))
        for row in ent.rows:
            for column_name in ent.pk:
                if row.get(column_name) is None:
                    problems.append("%s: primary key column %s is null"
                                    % (ent.fqn, column_name))
                    break

    for ent in entities.values():
        if not ent.rows:
            continue
        for fk in ent.fks:
            parent = entities.get(fk["parent"])
            if parent is None or not parent.pk:
                continue
            pool = key_pools.get(parent.fqn)
            if not pool:
                continue
            positions = []
            for parent_column in fk["parent_columns"]:
                positions.append(parent.pk.index(parent_column)
                                 if parent_column in parent.pk else 0)
            projected = set(tuple(key[p] for p in positions) for key in pool) \
                if positions != list(range(len(parent.pk))) else pool
            orphans = 0
            for row in ent.rows:
                values = tuple(row.get(c) for c in fk["columns"])
                if all(v is None for v in values):
                    continue
                if values not in projected:
                    orphans += 1
            if orphans:
                problems.append(
                    "%s: foreign key %s -> %s has %d row(s) with no parent key"
                    % (ent.fqn, "+".join(fk["columns"]), parent.fqn, orphans))

        for column in ent.columns:
            if column["nullable"]:
                continue
            nulls = sum(1 for row in ent.rows if row.get(column["name"]) is None)
            if nulls:
                problems.append("%s: NOT NULL column %s has %d null(s)"
                                % (ent.fqn, column["name"], nulls))
    return problems


# --------------------------------------------------------------------------------------
# write
# --------------------------------------------------------------------------------------

def write_entity(spark, entity):
    """Append the assembled rows using the table's own schema, so types cannot drift."""
    if not entity.rows:
        return 0
    schema = spark.sql("SELECT * FROM %s LIMIT 0" % entity.quoted).schema
    ordered = [tuple(row.get(field.name) for field in schema.fields) for row in entity.rows]
    frame = spark.createDataFrame(ordered, schema)
    frame.write.mode("append").saveAsTable("%s.%s.%s"
                                           % (entity.catalog, entity.schema, entity.table))
    return len(ordered)


def _llm_pools_for_model(spark, cfg, populate, rows, log):
    """Realistic value pools per table, bounded in time and reported as it goes.

    The pass is optional realism on top of a complete deterministic generator, so it is
    never allowed to decide how long an install takes: whatever has not answered inside
    the budget keeps its deterministic values. Progress is logged per wave, because a
    silent phase is indistinguishable from a hung one.
    """
    threads = max(1, cfg["threads"])
    pools_by_table = {}

    def _pools(entity):
        return entity.fqn, llm_value_pools(
            spark, entity, _llm_candidate_columns(entity),
            cfg["llm_endpoints"], rows, log)

    started = time.time()
    waves = -(-len(populate) // threads)
    budget = SAMPLE_LLM_TIMEOUT_S * waves
    pool_exec = ThreadPoolExecutor(max_workers=threads)
    try:
        pending = set(pool_exec.submit(_pools, e) for e in populate.values())
        answered, logged = 0, 0
        while pending:
            remaining = budget - (time.time() - started)
            if remaining <= 0:
                break
            done, pending = wait(pending, timeout=min(30.0, remaining))
            for future in done:
                answered += 1
                try:
                    fqn, pools = future.result()
                except Exception:
                    continue
                if pools:
                    pools_by_table[fqn] = pools
            if done and answered - logged >= threads:
                logged = answered
                log("  sample: LLM pools %d/%d table(s)  %.0fs"
                    % (answered, len(populate), time.time() - started))
        if pending:
            log("  sample: LLM pass hit its %ds budget with %d table(s) outstanding - "
                "those keep deterministic values" % (budget, len(pending)))
    finally:
        # Do not block the install on calls that are still in flight: they can only add
        # realism to tables that already have complete deterministic values.
        pool_exec.shutdown(wait=False, cancel_futures=True)
    log("  sample: LLM value pools for %d/%d table(s) in %.0fs"
        % (len(pools_by_table), len(populate), time.time() - started))
    return pools_by_table


# --------------------------------------------------------------------------------------
# entry point
# --------------------------------------------------------------------------------------

def generate_sample_data(spark, cfg, catalogs, log):
    """Populate every installed table. Returns a summary dict."""
    rows = cfg["rows"]
    seed = cfg["seed"]
    log("=" * 64)
    log("SAMPLE DATA: %d row(s) per table across %s" % (rows, ", ".join(catalogs)))
    entities = read_installed_model(spark, catalogs, log)
    populate = dict((k, e) for k, e in entities.items() if e.columns)
    if not populate:
        log("  sample: no installed tables found - nothing to populate")
        return {"tables": 0, "rows": 0, "problems": [], "written": 0}

    generate_all_keys(populate, rows, seed)

    pools_by_table = {}
    if cfg.get("llm") and cfg.get("llm_endpoints"):
        pools_by_table = _llm_pools_for_model(spark, cfg, populate, rows, log)

    for entity in populate.values():
        generate_rows(entity, populate, rows, seed, pools_by_table.get(entity.fqn))

    problems = assert_integrity(populate)
    if problems:
        log("  sample: INTEGRITY CHECK FAILED - %d problem(s), nothing written"
            % len(problems))
        for problem in problems[:20]:
            log("      - %s" % problem)
        raise Exception("Sample data integrity check failed: %s"
                        % "; ".join(problems[:5]))
    log("  sample: integrity check passed (unique keys, every foreign key resolves)")

    written, failures, done = [0], [], set()
    lock = threading.Lock()

    def _write(entity):
        try:
            count = write_entity(spark, entity)
            with lock:
                written[0] += count
                done.add(entity.fqn)
        except Exception as err:
            with lock:
                failures.append((entity.fqn, str(err).split("\n")[0][:200]))

    waves = write_order(populate)
    blocked = set()
    for depth, wave in enumerate(waves):
        due = [f for f in wave if f not in blocked]
        if due:
            with ThreadPoolExecutor(max_workers=cfg["threads"]) as writer:
                list(as_completed([writer.submit(_write, populate[f]) for f in due]))
        retry = [fqn for fqn, _ in failures if fqn in due]
        if retry:
            # The live coffee_roastery run lost one table to a transient write; one retry
            # costs a second and saves the whole downstream subtree from being skipped.
            log("  sample: retrying %d table(s) that failed to write in wave %d/%d"
                % (len(retry), depth + 1, len(waves)))
            failures[:] = [f for f in failures if f[0] not in retry]
            with ThreadPoolExecutor(max_workers=cfg["threads"]) as writer:
                list(as_completed([writer.submit(_write, populate[f]) for f in retry]))
        still_failed = set(fqn for fqn, _ in failures)
        if still_failed:
            blocked = _downstream_of(populate, still_failed)
            log("  sample: %d table(s) failed - skipping %d table(s) that reference them "
                "so nothing points at a row that was never written"
                % (len(still_failed), len(blocked - still_failed - done)))

    if failures:
        log("  sample: %d table(s) failed to write" % len(failures))
        for fqn, err in failures[:20]:
            log("      - %s -> %s" % (fqn, err))
        # Only a foreign-key cycle can put a child in the same wave as a failed parent,
        # so name any such table instead of letting the orphans go unreported.
        missing = set(populate) - done
        for fqn in sorted(done):
            broken = sorted(fk["parent"] for fk in populate[fqn].fks
                            if fk["parent"] in missing and fk["parent"] != fqn)
            if broken:
                log("      ! %s was written but references unwritten %s"
                    % (fqn, ", ".join(broken)))
    failures[:] = sorted(set(failures))
    log("SAMPLE DATA: wrote %d row(s) into %d table(s)%s"
        % (written[0], len(done),
           " (%d failed, %d skipped)"
           % (len(failures), len(populate) - len(done) - len(failures))
           if failures else ""))
    return {"tables": len(populate), "rows": rows, "written": written[0],
            "failed": [f[0] for f in failures],
            "skipped": sorted(set(populate) - done - set(f[0] for f in failures)),
            "problems": []}


## `main` - launch the job (or run the install)

If `session_id` is blank this cell launches the install as a Databricks job and prints
the run URL. When the job runs (session_id set) this same cell performs the install, so
all timestamped progress lands in this one cell's output. Earlier cells only define
functions and widgets.


In [ ]:
# === Install manifest + the Uninstall operation ===
# An uninstall must remove exactly what the matching install created and nothing else.
# Guessing from the catalog contents is unsafe: a user may install into a catalog that
# already holds their own schemas. So the install records what it created, and the
# uninstall reads that record back.

INSTALL_SCHEMA = "_install"   # holds the log volume and the manifest


def _manifest_path(cfg):
    return "/Volumes/%s/%s/logs/manifest_%s_%s.json" % (
        cfg["catalog"], INSTALL_SCHEMA, cfg["industry"], cfg["model_size"])


_SCHEMA_STMT_RE = _re.compile(
    r"CREATE\s+(?:SCHEMA|DATABASE)\s+(?:IF\s+NOT\s+EXISTS\s+)?"
    r"`?([^`.\s(]+)`?(?:\s*\.\s*`?([^`.\s(]+)`?)?", _re.I)


def plan_schemas(plan, default_catalog=None):
    """The (catalog, schema) pairs the plan's schema statements would create.

    The shipped models say CREATE DATABASE, not CREATE SCHEMA, and a statement may name
    the schema alone when the session already holds a catalog - so both spellings and
    both shapes are read here. Missing one is how an uninstall silently leaves the whole
    model behind, so a schema phase that parses to nothing raises rather than no-ops."""
    pairs = []
    for stmt in plan.get("schema", []):
        m = _SCHEMA_STMT_RE.search(stmt)
        if not m:
            continue
        catalog, schema = (m.group(1), m.group(2)) if m.group(2) else (default_catalog,
                                                                       m.group(1))
        if catalog and (catalog, schema) not in pairs:
            pairs.append((catalog, schema))
    if plan.get("schema") and not pairs:
        raise Exception(
            "Could not read a schema name out of any of the %d schema statement(s); "
            "refusing to continue, because that would drop or record nothing. First "
            "statement: %s" % (len(plan["schema"]), plan["schema"][0][:200]))
    return pairs


def installed_schemas(cfg, plan):
    """Every schema the install creates: the model's own, plus `_metrics` when metric
    views are on. `_install` is excluded - it holds the log sink, so uninstall drops it
    last, by hand, after the log has been flushed."""
    schemas = plan_schemas(plan, cfg.get("catalog"))
    if cfg["include_metrics"]:
        for cat in (cfg.get("target_catalogs") or [cfg["catalog"]]):
            if (cat, "_metrics") not in schemas:
                schemas.append((cat, "_metrics"))
    return schemas


def _prior_installer_created(catalog):
    """True iff a prior install of this catalog recorded `created_by_installer=True`.

    A repeated install on a catalog the installer previously created must not demote
    that flag. Otherwise `pre_existing` grows to include a catalog we made ourselves,
    the new manifest records `created_by_installer=False`, and the eventual widget
    Uninstall keeps the catalog behind (schemas dropped, catalog stranded).
    Alias: manifest-preserve-created-by-installer.
    """
    import glob as _glob
    logs_dir = "/Volumes/%s/%s/logs" % (catalog, INSTALL_SCHEMA)
    try:
        cands = sorted(_glob.glob("%s/manifest_*.json" % logs_dir))
    except Exception:
        return False
    for cand in cands:
        try:
            with open(cand, "r") as f:
                prior = json.loads(f.read())
        except Exception:
            continue
        for pc in prior.get("catalogs", []):
            if pc.get("name") == catalog and pc.get("created_by_installer"):
                return True
    return False


def write_install_manifest(cfg, plan, pre_existing, samples=None):
    """Record what this install created so a later uninstall can undo exactly that.

    `pre_existing` is the set of catalogs that already existed before the install ran;
    any target catalog outside it was created here and may therefore be dropped.
    A repeat install on a catalog we made ourselves inherits `created_by_installer=True`
    from the prior manifest via _prior_installer_created, so the eventual Uninstall
    still drops the catalog (alias: manifest-preserve-created-by-installer)."""
    catalogs_out = []
    for _c in (cfg.get("target_catalogs") or [cfg["catalog"]]):
        _created_here = _c not in pre_existing
        if not _created_here and _prior_installer_created(_c):
            _created_here = True
            log("  [manifest-preserve-created-by-installer FIRED] catalog `%s` "
                "inherits created_by_installer=True from a prior install manifest" % _c)
        catalogs_out.append({"name": _c, "created_by_installer": _created_here})
    body = {
        "industry": cfg["industry"],
        "model_size": cfg["model_size"],
        "version": cfg.get("resolved_version", "unknown"),
        "catalog": cfg["catalog"],
        "cataloging_style": cfg.get("cataloging_style", "One Catalog"),
        "include_metrics": bool(cfg["include_metrics"]),
        "installed_at": datetime.datetime.now().isoformat(timespec="seconds"),
        "catalogs": catalogs_out,
        "schemas": [[c, s] for c, s in installed_schemas(cfg, plan)],
        "samples": samples or {"enabled": False},
    }
    path = _manifest_path(cfg)
    try:
        with open(path, "w") as f:
            f.write(json.dumps(body, indent=2))
            f.flush()
            os.fsync(f.fileno())
        log("Install manifest: %s (%d schemas, %d catalog(s), %d created here)"
            % (path, len(body["schemas"]), len(body["catalogs"]),
               sum(1 for c in body["catalogs"] if c["created_by_installer"])))
    except Exception as e:
        log("Could not write the install manifest (%s) - an uninstall will fall back "
            "to the model plan and will not drop any catalog." % str(e)[:160])
    return body


def read_install_manifest(cfg):
    # Primary: the exact manifest path keyed by {catalog, industry, model_size}.
    exact = _manifest_path(cfg)
    try:
        with open(exact, "r") as f:
            return json.loads(f.read())
    except Exception as e:
        log("  [uninstall-manifest] exact manifest not read at %s (%s)"
            % (exact, str(e)[:120]))
    # Fallback: any manifest the install wrote into THIS catalog's _install/logs. A
    # custom (non-shipped) model uninstalled from the widget may carry a different
    # industry label than the install used, so match by catalog, not by filename.
    import glob as _glob
    logs_dir = "/Volumes/%s/%s/logs" % (cfg["catalog"], INSTALL_SCHEMA)
    for cand in sorted(_glob.glob("%s/manifest_*.json" % logs_dir)):
        try:
            with open(cand, "r") as f:
                body = json.loads(f.read())
            log("  [uninstall-manifest-catalog-scan FIRED] recovered manifest %s" % cand)
            return body
        except Exception:
            continue
    return None


def _existing_schemas(catalog):
    """Schema names present in a catalog right now, excluding UC's own."""
    try:
        rows = spark.sql(
            "SELECT schema_name FROM `%s`.information_schema.schemata" % catalog).collect()
    except Exception:
        return set()
    return set(r[0] for r in rows) - {"information_schema"}


def uninstall(cfg):
    """Drop exactly what the matching install created.

    Order matters: model schemas first, then `_install` (which holds the log sink), then
    any catalog the installer itself created. A catalog the installer did NOT create is
    always left in place, as is anything inside it that the install did not put there.
    Returns (failures, elapsed_seconds)."""
    run_start = time.time()
    manifest = read_install_manifest(cfg)
    if manifest:
        schemas = [(c, s) for c, s in manifest.get("schemas", [])]
        catalogs = manifest.get("catalogs", [])
        log("Manifest: %s (installed %s, version %s)"
            % (_manifest_path(cfg), manifest.get("installed_at"), manifest.get("version")))
    else:
        # No manifest: either the install predates them, or the volume is gone. The model
        # source still names the same schemas the install created, so drop those - but
        # leave every catalog alone, because nothing here proves the installer made it.
        log("No install manifest at %s - falling back to the model plan. No catalog "
            "will be dropped." % _manifest_path(cfg))
        plan = build_plan(cfg)
        schemas = installed_schemas(cfg, plan)
        catalogs = [{"name": c, "created_by_installer": False}
                    for c in (cfg.get("target_catalogs") or [cfg["catalog"]])]

    if not schemas:
        # Neither the manifest nor the model plan named anything to drop. If the
        # target catalog still holds model schemas, a silent SUCCESS would strand
        # them - fail loud (alias=uninstall-zero-schema-guard) and say how to fix it.
        leftover = sorted(_existing_schemas(cfg["catalog"]) - {INSTALL_SCHEMA})
        if leftover:
            raise Exception(
                "Uninstall resolved 0 schemas to drop but `%s` still holds %d schema(s): "
                "%s. The install manifest was not found and no model plan was available. "
                "Re-run Uninstall with 'local install' pointing at the model folder so "
                "the uninstall knows exactly what to remove."
                % (cfg["catalog"], len(leftover), ", ".join(leftover[:12])))
        log("Nothing to uninstall in `%s` (no recorded schemas; catalog already clean)."
            % cfg["catalog"])
    log("-" * 60)
    log("UNINSTALL %s/%s from `%s`: %d schema(s) across %d catalog(s)"
        % (cfg["industry"], cfg["model_size"], cfg["catalog"], len(schemas), len(catalogs)))
    for cat, schema in schemas:
        log("    drop `%s`.`%s`" % (cat, schema))

    failures = []
    stmts = ["DROP SCHEMA IF EXISTS `%s`.`%s` CASCADE" % (c, s) for c, s in schemas]
    for stmt, msg in run_phase("drop-schema", stmts, cfg["ddl_threads"], cfg["batch_size"]):
        failures.append(("drop-schema", stmt, msg))
    failures = [(ph, st, er) for ph, st, er in retry_failed(failures)]

    # The log sink lives in `_install`, so flush it before that schema disappears; the
    # rest of the uninstall is reported to the driver log and the job result.
    _flush_log_durable()
    _SINK["path"] = None
    for cat in sorted(set(c for c, _ in schemas) | set(c["name"] for c in catalogs)):
        try:
            spark.sql("DROP SCHEMA IF EXISTS `%s`.`%s` CASCADE" % (cat, INSTALL_SCHEMA))
        except Exception as e:
            failures.append(("drop-schema", "DROP SCHEMA `%s`.`%s`" % (cat, INSTALL_SCHEMA),
                             str(e)))

    dropped_catalogs = []
    for entry in catalogs:
        cat = entry["name"]
        if not entry.get("created_by_installer"):
            log("Keeping catalog `%s` - it existed before the install." % cat)
            continue
        left = _existing_schemas(cat) - {"default"}
        if left:
            # Something the install did not create is still in there. Dropping the
            # catalog would take it with us, so stop and say so.
            failures.append(("drop-catalog", "DROP CATALOG `%s`" % cat,
                             "catalog still holds schemas the install did not create: %s"
                             % ", ".join(sorted(left))))
            continue
        try:
            spark.sql("DROP CATALOG IF EXISTS `%s` CASCADE" % cat)
            dropped_catalogs.append(cat)
            log("Dropped catalog `%s` - the install created it." % cat)
        except Exception as e:
            failures.append(("drop-catalog", "DROP CATALOG `%s`" % cat, str(e)))

    # Prove it: re-read the catalogs that survive and confirm none of the schemas remain.
    residue = []
    for cat, schema in schemas:
        if cat in dropped_catalogs:
            continue
        if schema in _existing_schemas(cat):
            residue.append("`%s`.`%s`" % (cat, schema))
    if residue:
        failures.append(("verify", "post-uninstall check",
                         "still present: %s" % ", ".join(residue)))

    elapsed = time.time() - run_start
    log("=" * 64)
    log("UNINSTALL SUMMARY  %s/%s  from  `%s`"
        % (cfg["industry"], cfg["model_size"], cfg["catalog"]))
    log("  schemas dropped : %d" % (len(schemas) - len([f for f in failures
                                                        if f[0] == "drop-schema"])))
    log("  catalogs dropped: %d (%s)" % (len(dropped_catalogs),
                                         ", ".join(dropped_catalogs) or "none"))
    log("  failures        : %d" % len(failures))
    log("  total time      : %.1fs" % elapsed)
    return failures, elapsed


In [ ]:
# === main: launcher gate, then the full install (all timestamped logs below) ===
def _gen_session_id():
    import uuid
    return str((uuid.uuid4().int >> 64) & 9223372036854775807)


def _running_as_job():
    """True when this notebook is executing inside a Databricks job run (jobId set).
    Hard backstop so a launched job never re-launches itself, even if its session_id
    job parameter is somehow unreadable."""
    try:
        ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
        return bool(ctx.jobId().get())
    except Exception:
        return False


def launch_and_wait(cfg):
    """Launch this notebook as a tagged Databricks job, then BLOCK until the job
    finishes, emitting a timestamped liveness pulse every 20s. On
    completion, surface the job result and the Catalog Explorer URL. Returns
    (handled, success, exit_value)."""
    sid = _gen_session_id()
    try:
        version = "local" if cfg["local_install"] else latest_version(cfg)
    except Exception:
        version = "unknown"
    # Forward the resolved config to the job as base_parameters. The four UI widgets
    # travel under their widget names; the advanced settings (not widgets) travel as
    # plain job params and are read back via _wget in resolve_config.
    widgets = {
        "operation": "Uninstall" if cfg["operation"] == "uninstall" else "Install",
        "model": cfg["industry"],
        "model_size": cfg["model_size"],
        "catalog_name": cfg["catalog"],
        "cataloging_style": cfg.get("cataloging_style", "One Catalog"),
        "catalog_prefix": cfg.get("catalog_prefix", ""),
        "catalog_suffix": cfg.get("catalog_suffix", ""),
        "local_install": cfg.get("local_install_raw", cfg["local_install"]),
        "session_id": sid,
        "threads": str(cfg["threads"]),
        "batch_size": str(cfg["batch_size"]),
        "include_metrics": "true" if cfg["include_metrics"] else "false",
        "source_repo": cfg["source_repo"],
        "source_ref": cfg["source_ref"],
        "github_token": cfg["github_token"],
        "generate_samples": "Yes" if cfg["sample"]["enabled"] else "No",
        "sample_rows": str(cfg["sample"]["rows"]),
        "sample_seed": str(cfg["sample"]["seed"]),
        "sample_threads": str(cfg["sample"]["threads"]),
        "sample_llm": "true" if cfg["sample"]["llm"] else "false",
        "sample_llm_endpoints": ",".join(cfg["sample"]["llm_endpoints"]),
    }
    p = INSTALLER_TAG_PREFIX
    tags = {
        p + "source": "Data_Model_Installer",
        p + "industry": cfg["industry"],
        p + "size": cfg["model_size"],
        p + "version": version,
        p + "duration": "running",
        p + "session_id": sid,
    }
    nb_path = JobLauncher.get_current_notebook_path()
    if not nb_path:
        log("Could not determine notebook path; running the install directly instead.")
        return (False, False, None)
    job_name = "dbx_vibe_%s_%s_%s_%s" % (
        "uninstaller" if cfg["operation"] == "uninstall" else "installer",
        cfg["industry"], cfg["model_size"], version)
    log("No session_id -> launching the install as a Databricks job (%s) ..." % job_name)
    res = JobLauncher(nb_path, widgets, tags).launch(job_name=job_name, run_name=job_name)
    if not res["success"]:
        log("Job launch failed (%s) -> running the install directly instead." % res.get("error"))
        return (False, False, None)

    job_url = res["job_url"] or ("job_id=%s run_id=%s" % (res["job_id"], res["run_id"]))
    log("=" * 64)
    log("A Databricks JOB is installing %s/%s -> catalog `%s`." % (cfg["industry"], cfg["model_size"], cfg["catalog"]))
    log("Monitor it live here: %s" % job_url)
    log("Waiting for the job to finish (liveness pulse every 20s) ...")
    log("=" * 64)

    r = JobLauncher.wait_for_run(res["run_id"], job_url=job_url, pulse_seconds=20, logger=log)
    host, org = JobLauncher._get_workspace_context()
    catalog_url = ("%s/explore/data/%s%s" % (host, cfg["catalog"], ("?o=%s" % org if org else ""))) if host else ""
    nbout = r.get("notebook_output") or ""
    log("=" * 64)
    if r.get("result_state") == "SUCCESS":
        log("INSTALL JOB COMPLETE (%s)." % r.get("result_state"))
        if nbout:
            log("  Job result: %s" % nbout)
        if catalog_url:
            log("  Open the catalog here: %s" % catalog_url)
        log("=" * 64)
        exitval = "SUCCESS via job: %s/%s -> `%s` | catalog: %s | job: %s" % (
            cfg["industry"], cfg["model_size"], cfg["catalog"], catalog_url or "(n/a)", job_url)
        return (True, True, exitval)
    log("INSTALL JOB DID NOT SUCCEED (state=%s, result=%s)." % (r.get("life_cycle_state"), r.get("result_state")))
    if nbout:
        log("  Job result: %s" % nbout)
    if r.get("error"):
        log("  Job error: %s" % str(r.get("error"))[:300])
    log("  Inspect the run here: %s" % job_url)
    log("=" * 64)
    exitval = "FAILED via job: %s/%s (result=%s) | job: %s" % (
        cfg["industry"], cfg["model_size"], r.get("result_state"), job_url)
    return (True, False, exitval)


def install(cfg, plan):
    """Apply the plan phase-by-phase, retry failures serially, summarize.

    Returns (final_failures, elapsed_seconds, timings) where timings is an
    OrderedDict {phase: seconds} so the breakdown can be surfaced in the exit value."""
    from collections import OrderedDict as _OD
    run_start = time.time()
    threads, bs = cfg["threads"], cfg["batch_size"]
    catalog = cfg["catalog"]
    failures = []
    timings = _OD()

    # Metadata-mutation DDL (ADD CONSTRAINT, SET TAGS) goes through the Unity Catalog
    # API. At 32 concurrent threads UC throttles and returns 504/UC_CLIENT_EXCEPTION
    # after a few hundred calls, which collapses throughput. Cap these phases to a
    # UC-friendly concurrency; object-creation phases (table/metric) keep full threads.
    ddl_threads = cfg["ddl_threads"]

    def do(phase, stmts, bsize, group=False, serial=False, workers=None):
        t = time.time()
        w = workers if workers is not None else threads
        n = 0
        for stmt, msg in run_phase(phase, stmts, w, bsize, group=group, serial=serial):
            failures.append((phase, stmt, msg))
            n += 1
        timings[phase] = time.time() - t
        return n

    log("-" * 60)
    # Catalog is the prerequisite for every other statement. If it cannot be created
    # (e.g. a misconfigured metastore with no storage root), abort NOW with a clear
    # error instead of grinding through thousands of doomed schema/table/fk/tag calls.
    # Catalog creation is metastore-config-sensitive (Default Storage / no
    # storage root). Use the robust helper, then DO NOT run the plan's plain
    # CREATE CATALOG, which fails on Default-Storage-only metastores even when
    # the catalog already exists (the storage-root check precedes IF NOT EXISTS).
    target_catalogs = cfg.get("target_catalogs") or [catalog]
    cat_fail = 0
    for tc in target_catalogs:
        try:
            _ensure_catalog(tc, log)
        except Exception as _ce:
            failures.append(("catalog", "CREATE CATALOG `%s`" % tc, str(_ce)))
            cat_fail += 1
    missing = [tc for tc in target_catalogs if not _catalog_exists(tc)]
    if cat_fail or missing:
        err = failures[-1][2] if failures else ("missing catalogs: %s" % missing)
        elapsed = time.time() - run_start
        msg = ("FATAL: required catalog(s) could not be created - aborting before the "
               "remaining %d statements. Root cause: %s"
               % (sum(len(v) for k, v in plan.items() if k != "catalog"),
                  " ".join((err or "").split())[:240]))
        log("=" * 64)
        log(msg)
        raise Exception(msg)
    metrics_catalog = catalog
    schema_stmts = list(plan["schema"])
    if cfg["include_metrics"]:
        schema_stmts.append("CREATE SCHEMA IF NOT EXISTS `%s`.`_metrics`" % metrics_catalog)
    do("schema", schema_stmts, bs)
    do("table", plan["table"], bs)
    do("fk", plan["fk"], bs, group=True, workers=ddl_threads)
    do("tag", plan["tag"], bs, group=True, workers=ddl_threads)
    if cfg["include_metrics"]:
        do("metric", plan["metric"], bs)
    if plan["other"]:
        do("other", plan["other"], bs)

    log("First-pass failures (pre-retry): %d" % len(failures))
    t_retry = time.time()
    final = retry_failed(failures)
    timings["retry"] = time.time() - t_retry
    elapsed = time.time() - run_start

    log("=" * 64)
    log("INSTALL SUMMARY  %s/%s  ->  catalog `%s`" % (cfg["industry"], cfg["model_size"], catalog))
    for k, v in plan.items():
        if v:
            log("  planned %-8s %d" % (k, len(v)))
    log("  phase timings: " + "  ".join("%s=%.0fs" % (k, v) for k, v in timings.items()))
    log("  total time: %.1fs (%.1f min)" % (elapsed, elapsed / 60.0))
    log("  unrecoverable failures: %d  (enumerated below by caller)" % len(final))
    return final, elapsed, timings


# _SINK is defined in the helpers cell (next to _LOG_BUFFER) so log() can see it.


def setup_log_sink(cfg):
    """Create <catalog>._install.logs Volume and mirror the live log there every few
    seconds, so the full run log is retrievable via CLI during and after the run.
    Best effort - if it cannot be created, console/heartbeat logging still works."""
    cat = cfg["catalog"]
    try:
        _ensure_catalog(cat, log)
        spark.sql("CREATE SCHEMA IF NOT EXISTS `%s`.`_install`" % cat)
        spark.sql("CREATE VOLUME IF NOT EXISTS `%s`.`_install`.`logs`" % cat)
    except Exception as e:
        log("Log sink unavailable (%s) - console logging only" % str(e)[:160])
        return
    ts = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
    path = "/Volumes/%s/_install/logs/install_%s_%s_%s.log" % (cat, cfg["industry"], cfg["model_size"], ts)
    # Seed the file with everything logged so far, then switch log() to inline append.
    try:
        with _LOG_LOCK:
            seed = "\n".join(_LOG_BUFFER)
            with open(path, "w") as f:
                f.write(seed + "\n")
            _SINK["path"] = path
    except Exception as e:
        log("Log sink unavailable (%s) - console logging only" % str(e)[:160])
        return
    log("Log sink: %s" % path)


def teardown_log_sink():
    # Inline append already persisted every line; nothing to flush.
    _SINK["path"] = None


def write_failures_manifest(cfg, final):
    """Persist unrecoverable statements (phase, error, SQL) to the install volume so a
    user can review/replay them after the run. Best effort."""
    cat = cfg["catalog"]
    ts = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
    path = "/Volumes/%s/_install/logs/failures_%s_%s_%s.json" % (cat, cfg["industry"], cfg["model_size"], ts)
    try:
        rows = [{"phase": ph, "error": err, "sql": stmt} for ph, stmt, err in final]
        with open(path, "w") as f:
            f.write(json.dumps(rows, indent=2))
            f.flush()
            os.fsync(f.fileno())   # durable before dbutils.notebook.exit() terminates us
        log("Failures manifest: %s" % path)
    except Exception as e:
        log("Could not write failures manifest (%s)" % str(e)[:160])


def main():
    # First-run guard: an industry must be chosen explicitly. The widget defaults to the
    # SELECT_PROMPT placeholder, so on the very first Run All nothing is selected yet -
    # print a friendly prompt and stop cleanly (no error, no job launched).
    if (_wget("operation", "Install").strip().lower() != "uninstall"
            and _wget("model", "").strip() not in INDUSTRIES
            and not _wget("local_install", "").strip()):
        log("Please select an industry in the 'industry' widget (or point 'local install' "
            "at a model folder), then click Run All again.")
        return

    cfg = resolve_config()
    log("Operation     : %s" % cfg["operation"])
    log("Mode          : %s" % cfg["mode"])
    log("Industry      : %s" % cfg["industry"])
    log("Model size    : %s" % cfg["model_size"])
    log("Target catalog: %s" % cfg["catalog"])
    log("Cataloging    : %s (prefix=%r suffix=%r)" % (
        cfg.get("cataloging_style", "One Catalog"), cfg.get("catalog_prefix", ""), cfg.get("catalog_suffix", "")))
    log("Threads       : %d   Batch size: %d" % (cfg["threads"], cfg["batch_size"]))
    log("Metric views  : %s" % cfg["include_metrics"])
    log("Samples       : %s%s" % (
        "yes" if cfg["sample"]["enabled"] else "no",
        " (%d rows/table)" % cfg["sample"]["rows"] if cfg["sample"]["enabled"] else ""))

    # Launcher gate: an interactive run with no session_id launches the install as a
    # Databricks job. The _running_as_job() backstop guarantees a launched job runs the
    # install in-place instead of re-launching itself.
    if not cfg["session_id"] and not _running_as_job():
        handled, ok, exitval = launch_and_wait(cfg)
        if handled:
            dbutils.notebook.exit(exitval)
            return

    # Running in-place (the launched job, or launch was unavailable). Keep the log
    # sink OPEN through the final verdict + manifest so they land in the Volume log;
    # _flush_log_durable() fsyncs the last state before exit, then teardown.
    if cfg["operation"] == "uninstall":
        # No volume log sink here: it lives in `_install`, inside the catalog being
        # removed, so it would be deleted by the very operation it is recording.
        failures, elapsed = uninstall(cfg)
        if failures:
            for phase, stmt, err in failures:
                log("  [%s] %s" % (phase, " ".join(stmt.split())[:110]))
                log("        -> %s" % (" ".join((err or "").split())[:200]))
            raise Exception("Uninstall FAILED: %d statement(s) unrecoverable in `%s`"
                            % (len(failures), cfg["catalog"]))
        result = ("UNINSTALLED: %s/%s from `%s` (%.1f min)"
                  % (cfg["industry"], cfg["model_size"], cfg["catalog"], elapsed / 60.0))
        log(result)
        dbutils.notebook.exit(result)
        return

    # Probed BEFORE the log sink is set up: the sink lives in the target catalog and
    # creates it when missing, so a later probe would see a catalog this install made
    # and record it as pre-existing - and the uninstall would then refuse to drop it.
    probed = set(cfg.get("target_catalogs") or [cfg["catalog"]])
    pre_existing = set(c for c in probed if _catalog_exists(c))

    setup_log_sink(cfg)
    result = None
    try:
        plan = build_plan(cfg)
        # A per-division / per-domain layout only names its catalogs once the plan is
        # built. None of them has been created yet, so this is still a true "before".
        for _c in (cfg.get("target_catalogs") or []):
            if _c not in probed:
                probed.add(_c)
                if _catalog_exists(_c):
                    pre_existing.add(_c)
        final, elapsed, timings = install(cfg, plan)

        # Samples run only on a structurally clean install: populating tables whose
        # keys or foreign keys failed to apply would write rows that cannot be joined.
        sample_note = ""
        sample_summary = {"enabled": bool(cfg["sample"]["enabled"])}
        if cfg["sample"]["enabled"]:
            if [f for f in final if f[0] != "metric"]:
                log("Samples skipped: the install has structural failures.")
                sample_note = " | samples: skipped (structural failures)"
            else:
                t_samples = time.time()
                summary = generate_sample_data(
                    spark, cfg["sample"], cfg.get("target_catalogs") or [cfg["catalog"]], log)
                timings["samples"] = time.time() - t_samples
                sample_note = (" | samples: %d rows in %d tables"
                               % (summary["written"], summary["tables"]))
                if summary.get("failed"):
                    sample_note += " (%d failed)" % len(summary["failed"])
                sample_summary.update(rows=cfg["sample"]["rows"], tables=summary["tables"],
                                      written=summary["written"],
                                      failed=len(summary.get("failed") or []))

        # Written after the samples so the manifest reflects the finished install.
        write_install_manifest(cfg, plan, pre_existing, sample_summary)

        # Tag the job with the final install duration (best effort; only when run as a job).
        p = INSTALLER_TAG_PREFIX
        tag_res = JobLauncher.update_job_tags({
            p + "industry": cfg["industry"],
            p + "size": cfg["model_size"],
            p + "version": cfg.get("resolved_version", "unknown"),
            p + "duration": "%.1fmin" % (elapsed / 60.0),
        })
        log("Tag update: %s" % ("ok" if tag_res["success"] else tag_res.get("error")))

        timing_str = " ".join("%s=%.0fs" % (k, v) for k, v in timings.items())
        total = sum(len(v) for v in plan.values())

        if final:
            # Structural phases must be perfect; metric views are source-model SQL and a
            # bad column reference there is the model author's defect, not an install
            # fault. So: hard-fail on any structural failure, warn on metric-only ones.
            structural = [f for f in final if f[0] != "metric"]
            log("=" * 64)
            log("UNRECOVERABLE STATEMENTS: %d  (structural=%d, metric=%d)"
                % (len(final), len(structural), len(final) - len(structural)))
            for phase, stmt, err in final:
                log("  [%s] %s" % (phase, " ".join(stmt.split())[:110]))
                log("        -> %s" % (" ".join((err or "").split())[:200]))
            write_failures_manifest(cfg, final)   # persisted to <catalog>._install volume
            if structural:
                raise Exception(
                    "Install FAILED: %d structural statement(s) unrecoverable in `%s` "
                    "(+%d metric-view source defects) | timings: %s"
                    % (len(structural), cfg["catalog"], len(final) - len(structural), timing_str))
            result = ("INSTALLED_WITH_WARNINGS: %s/%s -> `%s` (%d statements, %d metric-view "
                      "source defects, 0 structural failures, %.1f min) | timings: %s | log: %s"
                      % (cfg["industry"], cfg["model_size"], cfg["catalog"], total,
                         len(final), elapsed / 60.0, timing_str, _SINK["path"])) + sample_note
            log(result)
        else:
            result = ("SUCCESS: %s/%s -> `%s` (%d statements, 0 failures, %.1f min) | timings: %s | log: %s"
                      % (cfg["industry"], cfg["model_size"], cfg["catalog"],
                         total, elapsed / 60.0, timing_str, _SINK["path"])) + sample_note
            log(result)
    finally:
        _flush_log_durable()   # fsync the complete log (incl. final verdict) before exit
        teardown_log_sink()

    dbutils.notebook.exit(result)


main()
